In [2]:
import warnings; warnings.filterwarnings("ignore")
import re, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import shap

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import KNNImputer
from sklearn.model_selection import (StratifiedKFold,
                                     train_test_split, RandomizedSearchCV,
                                     cross_val_predict)
from sklearn.metrics import (classification_report, accuracy_score,
                              roc_auc_score, confusion_matrix,
                              precision_score, recall_score, f1_score)
from imblearn.over_sampling import BorderlineSMOTE
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("⚠ XGBoost not installed — skipping XGBoost model")

SEED = 42
np.random.seed(SEED)

# ── Palette ──────────────────────────────────────────────────────────────────
P = {"bg":"#0F1117","card":"#1A1D27","accent":"#7C6AF7","green":"#4ADE80",
     "red":"#F87171","text":"#E8E8F0","muted":"#8B8FA8","border":"#2A2D3E",
     "orange":"#FB923C","blue":"#60A5FA","yellow":"#FCD34D"}

plt.rcParams.update({
    "figure.facecolor":P["bg"],"axes.facecolor":P["card"],
    "axes.edgecolor":P["border"],"axes.labelcolor":P["text"],
    "xtick.color":P["muted"],"ytick.color":P["muted"],"text.color":P["text"],
    "grid.color":P["border"],"grid.linestyle":"--","grid.alpha":0.4,
    "font.family":"sans-serif","axes.titlesize":11,"axes.labelsize":9,
})

# ── Non-traditional features (paper novelty) ─────────────────────────────────
NON_TRADITIONAL = {
    "hb"             : "Haemoglobin (g/dL)",
    "albumin "       : "Serum Albumin (g/dL)",
    "ALP"            : "Alkaline Phosphatase (U/L)",
    "HBA1C"          : "HbA1c (%)",
    "bill "          : "Serum Bilirubin (mg/dL)",
    "BLOOD LOSS "    : "Intra-op Blood Loss (mL)",
    "CBD DIAMETER"   : "CBD Diameter (mm)",
    "ca 19-9"        : "CA 19-9 (U/mL)",
    "Cea"            : "CEA (ng/mL)",
    "fever "         : "Pre-op Fever",
    "RECENT DIABETES": "Recent Diabetes",
    "comorbidity"    : "Comorbidity",
    "neoadjuvant "   : "Neoadjuvant Treatment",
    "ercp stenting " : "ERCP Stenting",
    "jaundice"       : "Obstructive Jaundice",
    "pain "          : "Pre-op Pain",
    "weight loss "   : "Weight Loss",
    "bleeding "      : "Pre-op Bleeding",
    "Age "           : "Patient Age",
    "Sex"            : "Patient Sex",
}

# =============================================================================
# STEP 1 — LOAD & CLEAN
# =============================================================================
df = pd.read_excel("D:\\medical_1D\\1D data 2.xlsx")
print(f"Raw: {df.shape}")

DROP = ["DGE","PPH","SSI","CLAVIEN DINDO","BILE LAK","HOSPITAL STAY ",
        "READMISION","RESURGERY","mortality",
        "s no ","Name ","Address","CR Number ",
        "Unnamed: 5","Unnamed: 6","Unnamed: 7",
        "date of admisson ","date of sx ","date of discharge ","date of surgery",
        "cect","eus ","periampullary carcinoma ","surgery","CHROMAGANIN "]
df.drop(columns=[c for c in DROP if c in df.columns], inplace=True)

df.dropna(subset=["POPF"], inplace=True)
df["POPF"] = (df["POPF"] != 0).astype(int)

def auto_encode(col, series):
    name = col.strip().lower()
    if series.dtype in [np.float64, np.int64, float, int]:
        num = pd.to_numeric(series, errors="coerce")
        return num if num.notna().mean() >= 0.4 else None
    s = series.astype(str).str.strip()
    if name == "lap open": return None
    if name in ("fever ", "neoadjuvant "):
        def _b(v):
            sv = str(v).strip()
            if sv in ("-","9","nan",""): return np.nan
            try:   return 1.0 if float(sv) > 0 else 0.0
            except: return np.nan
        return series.apply(_b)
    if name == "ercp stenting ":
        return series.apply(lambda v: 1.0 if str(v).strip().startswith("1")
                            else (0.0 if str(v).strip() == "0" else np.nan))
    if name == "comorbidity":
        return series.apply(lambda v: 0.0 if str(v).strip() in ("0","nan","")
                            else (0.0 if str(v).strip() == "0" else 1.0))
    if name == "bill ":
        return pd.to_numeric(s.str.replace(r"[Oo]","0",regex=True), errors="coerce")
    if name == "cea":
        return pd.to_numeric(s.str.replace("`",""), errors="coerce")
    if name == "tumor size":
        return series.apply(lambda v: max(
            (float(n) for n in re.findall(r"[\d.]+", str(v))), default=np.nan))
    num = pd.to_numeric(s.str.replace("`","").str.replace(",","."), errors="coerce")
    return num if num.notna().mean() >= 0.4 else None

y   = df["POPF"].astype(int)
raw = df.drop(columns=["POPF"])
keep = {}
for col in raw.columns:
    res = auto_encode(col, raw[col])
    if res is not None and res.notna().mean() >= 0.20 and res.nunique() > 1:
        keep[col] = res.astype(float)

X_df = pd.DataFrame(keep)
feature_names = list(X_df.columns)

def is_binary_col(series):
    vals = set(series.dropna().unique())
    return vals.issubset({0.0, 1.0})

binary_cols     = [c for c in X_df.columns if is_binary_col(X_df[c])]
continuous_cols = [c for c in X_df.columns if c not in binary_cols]

# ── IMPROVEMENT 1: KNN Imputer for smarter missing value filling ──────────────
print("\n── KNN Imputation (k=5) for missing values ──")
knn_imp = KNNImputer(n_neighbors=5, weights="distance")

# Impute continuous cols with KNN, binary with mode (preserves 0/1 integrity)
X_cont = X_df[continuous_cols].values
X_cont_imputed = knn_imp.fit_transform(X_cont)
for i, col in enumerate(continuous_cols):
    X_df[col] = X_cont_imputed[:, i]
for col in binary_cols:
    X_df[col].fillna(X_df[col].mode()[0], inplace=True)

print(f"Binary columns  (mode-imputed) : {len(binary_cols)}")
print(f"Continuous cols (KNN-imputed)  : {len(continuous_cols)}")

X_np = X_df.values.copy()
y_np = y.values.copy()
binary_idx     = [feature_names.index(c) for c in binary_cols]
continuous_idx = [feature_names.index(c) for c in continuous_cols]
print(f"\nOriginal: {len(y_np)} patients | POPF: {y_np.sum()}  No-POPF: {(y_np==0).sum()}")

# =============================================================================
# STEP 2 — 80/20 TRAIN/TEST SPLIT (BEFORE AUGMENTATION — no leakage)
# =============================================================================
print("\n" + "="*55)
print("STEP 2 — 80/20 TRAIN/TEST SPLIT (before augmentation)")
print("="*55)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_np, y_np, test_size=0.20, stratify=y_np, random_state=SEED)
print(f"Train (real): {len(y_train_r)} patients | POPF={y_train_r.sum()}  No-POPF={(y_train_r==0).sum()}")
print(f"Test  (real): {len(y_test_r)}  patients | POPF={y_test_r.sum()}  No-POPF={(y_test_r==0).sum()}")
print("  Test set = real patients only, never used during augmentation or tuning")

# =============================================================================
# STEP 3 — BINARY-SAFE INTERPOLATION AUGMENTATION (training set only)
# ── IMPROVEMENT 2: TARGET_N raised 200→300 for more robust training ──────────
# =============================================================================
print("\n── Linear interpolation augmentation (train only) ──")

TARGET_N = 300                      # ← raised from 200
cont_idx = [i for i in range(X_np.shape[1]) if i not in binary_idx]
rng_aug  = np.random.default_rng(SEED)

def linear_interpolate_class(X_class, n_needed, binary_idx, cont_idx, rng):
    n_real, n_feat = X_class.shape
    X_syn = np.empty((n_needed, n_feat), dtype=float)
    for s in range(n_needed):
        a, b = rng.choice(n_real, size=2, replace=(n_real < 2))
        lam  = rng.uniform(0.0, 1.0)
        X_syn[s, cont_idx]   = lam * X_class[a, cont_idx] + (1.0-lam) * X_class[b, cont_idx]
        X_syn[s, binary_idx] = np.where(lam >= 0.5,
                                         X_class[a, binary_idx],
                                         X_class[b, binary_idx])
    return X_syn

idx_pos = np.where(y_train_r == 1)[0]
idx_neg = np.where(y_train_r == 0)[0]
X_pos, X_neg = X_train_r[idx_pos], X_train_r[idx_neg]
half      = TARGET_N // 2
n_syn_pos = max(0, half - len(X_pos))
n_syn_neg = max(0, half - len(X_neg))
X_syn_pos = linear_interpolate_class(X_pos, n_syn_pos, binary_idx, cont_idx, rng_aug)
X_syn_neg = linear_interpolate_class(X_neg, n_syn_neg, binary_idx, cont_idx, rng_aug)

X_aug = np.vstack([X_train_r, X_syn_pos, X_syn_neg])
y_aug = np.concatenate([y_train_r,
                        np.ones(n_syn_pos, dtype=int),
                        np.zeros(n_syn_neg, dtype=int)])

print(f"Real train : {len(X_train_r)}  |  Augmented train : {len(X_aug)}  |  Test (real) : {len(X_test_r)}")

for idx in binary_idx:
    vals = set(np.unique(X_aug[:, idx]))
    assert vals.issubset({0.0, 1.0}), f"Binary violated: {feature_names[idx]}"
print("  Binary column integrity verified")

# =============================================================================
def geo_optimizer(X_data, y_data, n_agents, n_iter,
                  step_size=0.3, explore_prob=0.2,
                  min_feat=3, label="GEO"):

    n_feat = X_data.shape[1]
    rng = np.random.default_rng(SEED)

    # ── Fitness (same as before: AUC-based) ──────────────────────────
    def fitness(mask):
        if mask.sum() == 0:
            return 0.0

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        aucs = []

        for tr, te in skf.split(X_data[:, mask], y_data):
            Xtr, Xte = X_data[tr][:, mask], X_data[te][:, mask]
            ytr, yte = y_data[tr], y_data[te]

            k = min(4, int(ytr.sum()) - 1)
            if k < 1:
                continue

            try:
                Xs, ys = BorderlineSMOTE(random_state=SEED, k_neighbors=k).fit_resample(Xtr, ytr)
            except:
                Xs, ys = Xtr, ytr

            model = RandomForestClassifier(
                n_estimators=50,
                max_depth=5,
                class_weight="balanced",
                random_state=SEED
            )
            model.fit(Xs, ys)

            prob = model.predict_proba(Xte)[:, 1]
            if len(np.unique(yte)) > 1:
                aucs.append(roc_auc_score(yte, prob))

        return float(np.mean(aucs)) if aucs else 0.0

    # ── Ensure minimum features ──────────────────────────────────────
    def enforce_min(mask):
        if mask.sum() < min_feat:
            off = np.where(mask == 0)[0]
            need = min_feat - int(mask.sum())
            mask[rng.choice(off, min(need, len(off)), replace=False)] = 1
        return mask

    # ── Initialize population ────────────────────────────────────────
    positions = rng.random((n_agents, n_feat))  # continuous [0,1]
    masks = (positions > 0.5).astype(float)
    masks = np.array([enforce_min(m) for m in masks])

    scores = np.array([fitness(m.astype(bool)) for m in masks])

    best_idx = np.argmax(scores)
    g_best = masks[best_idx].copy()
    g_score = scores[best_idx]

    history = [g_score]
    print(f"  [{label}] Init best={g_score:.4f} features={int(g_best.sum())}")

    # ── GEO Main Loop ────────────────────────────────────────────────
    for it in range(n_iter):

        for i in range(n_agents):

            if rng.random() < explore_prob:
                # Random exploration
                new_pos = rng.random(n_feat)
            else:
                # Geometric move toward best
                direction = g_best - masks[i]
                step = step_size * direction
                noise = rng.normal(0, 0.1, n_feat)

                new_pos = masks[i] + step + noise

            # Convert to binary
            new_mask = (new_pos > 0.5).astype(float)
            new_mask = enforce_min(new_mask)

            sc = fitness(new_mask.astype(bool))

            # Greedy update
            if sc > scores[i]:
                masks[i] = new_mask
                scores[i] = sc

                if sc > g_score:
                    g_best = new_mask.copy()
                    g_score = sc

        history.append(g_score)

        if (it + 1) % 5 == 0 or it == 0:
            print(f"  [{label}] Iter {it+1}/{n_iter} best={g_score:.4f} features={int(g_best.sum())}")

    print(f"  [{label}] Done best={g_score:.4f} features={int(g_best.sum())}")
    return g_best.astype(bool), g_score, history

print("\n" + "="*55)
print("STAGE 1 — aco on all features  [n_bees=15, n_iter=20, AUC fitness]")
print("="*55)
mask_A, score_A, hist_A = geo_optimizer(
    X_aug, y_aug,
    n_agents=15,
    n_iter=20,
    step_size=0.3,
    explore_prob=0.2,
    min_feat=4,
    label="GEO Stage-1"
)
set_A = [feature_names[i] for i, m in enumerate(mask_A) if m]
set_B = [feature_names[i] for i, m in enumerate(mask_A) if not m]
print(f"\n  Set A (important)  : {len(set_A)}  ->  {set_A}")
print(f"  Set B (unimportant): {len(set_B)}  ->  {set_B}")

print("\n" + "="*55)
print("STAGE 2 — aco on unimportant Set B  [n_bees=12, n_iter=15]")
print("="*55)

idx_B = [i for i, m in enumerate(mask_A) if not m]

mask_C, score_C, hist_C = geo_optimizer(
    X_aug[:, idx_B], y_aug,
    n_agents=12,
    n_iter=15,
    step_size=0.4,
    explore_prob=0.25,
    min_feat=2,
    label="GEO Stage-2"
)
set_C = [set_B[i] for i, m in enumerate(mask_C) if m]
print(f"\n  Set C (rescued): {len(set_C)}  ->  {set_C}")

final_features = list(dict.fromkeys(set_A + set_C))
print(f"\nFinal features (A + C): {len(final_features)}")
for f in final_features:
    tag = "  [NON-TRAD]" if f in NON_TRADITIONAL else ""
    src = "A" if f in set_A else "C"
    print(f"  [{src}]  {f}{tag}")

# =============================================================================
# STEP 5 — MULTI-MODEL TRAINING + DEEP HYPERPARAMETER TUNING
# ── IMPROVEMENT 4: RobustScaler pipelines, BorderlineSMOTE, Youden threshold,
#                   n_iter=100, wider grids, RepeatedStratifiedKFold ──────────
# =============================================================================
feat_idx = [feature_names.index(f) for f in final_features]
X_tr     = X_aug[:, feat_idx].copy()
y_tr     = y_aug.copy()
X_te     = X_test_r[:, feat_idx].copy()
y_te     = y_test_r.copy()

# ── Feature scaling on continuous columns only ────────────────────────────────
final_cont_mask = np.array([f not in binary_cols for f in final_features])
scaler = RobustScaler()

def scale_data(X_train, X_test, cont_mask):
    """Fit scaler on train continuous cols, transform both — no leakage."""
    X_tr_s = X_train.copy()
    X_te_s = X_test.copy()
    if cont_mask.any():
        X_tr_s[:, cont_mask] = scaler.fit_transform(X_train[:, cont_mask])
        X_te_s[:, cont_mask] = scaler.transform(X_test[:, cont_mask])
    return X_tr_s, X_te_s

X_tr_scaled, X_te_scaled = scale_data(X_tr, X_te, final_cont_mask)

# ── BorderlineSMOTE on augmented training set ─────────────────────────────────
k_sm = min(4, int(y_tr.sum()) - 1)
try:
    Xs_tr, ys_tr = BorderlineSMOTE(random_state=SEED, k_neighbors=k_sm).fit_resample(
        X_tr_scaled, y_tr)
except Exception:
    from imblearn.over_sampling import SMOTE
    Xs_tr, ys_tr = SMOTE(random_state=SEED, k_neighbors=k_sm).fit_resample(
        X_tr_scaled, y_tr)

neg_count = int((ys_tr == 0).sum())
pos_count = int((ys_tr == 1).sum())
spw       = round(neg_count / max(pos_count, 1), 2)
print(f"\nBorderlineSMOTE output -> neg={neg_count}  pos={pos_count}  scale_pos_weight={spw}")

print("\n" + "="*55)
print("STEP 5 — MULTI-MODEL TRAINING + DEEP HYPERPARAMETER TUNING")
print(f"  Train : {len(Xs_tr)} (BorderlineSMOTE on augmented) | Test : {len(X_te_scaled)} (real)")
print(f"  Scaling: RobustScaler on {final_cont_mask.sum()} continuous features")
print("="*55)

# ── IMPROVEMENT 5: Youden's J threshold (sensitivity + specificity − 1) ──────
def best_threshold_youden(model, X, y):
    """
    Find threshold maximising Youden's J = Sensitivity + Specificity − 1.
    More robust than F1-only for imbalanced medical data.
    Note: cross_val_predict requires partitions so we use StratifiedKFold(10).
    """
    skf      = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
    cv_prob  = cross_val_predict(model, X, y, cv=skf, method="predict_proba")[:, 1]
    thresholds = np.arange(0.20, 0.81, 0.02)
    best_t, best_j = 0.5, -1.0
    for t in thresholds:
        yp  = (cv_prob >= t).astype(int)
        rec = recall_score(y, yp, zero_division=0)                    # sensitivity
        tn  = confusion_matrix(y, yp).ravel()
        # specificity = TN / (TN + FP)
        cm  = confusion_matrix(y, yp)
        if cm.shape == (2, 2):
            spec = cm[0, 0] / max(cm[0, 0] + cm[0, 1], 1)
        else:
            spec = 0.0
        j = rec + spec - 1.0
        if j > best_j:
            best_j = j
            best_t = t
    return float(best_t)

# ── Model definitions ─────────────────────────────────────────────────────────
MODELS = {}

# 1. Random Forest — wider grid, more estimators
MODELS["Random Forest"] = (
    RandomForestClassifier(random_state=SEED),
    {
        "n_estimators": [500, 700, 1000],
        "max_depth": [6, 8, 10, 12, None],
        "min_samples_leaf"  : [1, 2],
        "min_samples_split" : [2,3],
        "max_features"      : ["sqrt",0.7,None],
        "class_weight"      : ["balanced", "balanced_subsample",
                               {0:1,1:2}, {0:1,1:3}, {0:1,1:4}],
        "max_samples"       : [0.7, 0.8, 0.9, None],
    }
)

# 2. Decision Tree
MODELS["Decision Tree"] = (
    DecisionTreeClassifier(random_state=SEED),
    {
        "max_depth"         : [3, 4, 5, 6, 7, 8, None],
        "min_samples_leaf"  : [1, 2, 3, 5],
        "min_samples_split" : [2, 4, 6],
        "criterion"         : ["gini", "entropy"],
        "class_weight"      : ["balanced", {0:1,1:2}, {0:1,1:3}, {0:1,1:4}],
        "ccp_alpha"         : [0.0, 0.002, 0.005, 0.01, 0.02, 0.05],
        "splitter"          : ["best", "random"],
    }
)

# 3. XGBoost — deeper regularisation grid
if HAS_XGB:
    MODELS["XGBoost"] = (
        XGBClassifier(random_state=SEED, eval_metric="logloss", verbosity=0),
        {
            "n_estimators"     : [100, 200, 300, 500, 700],
            "max_depth"        : [3, 4, 5, 6, 7],
            "learning_rate"    : [0.005, 0.01, 0.05, 0.1, 0.2],
            "subsample"        : [0.6, 0.7, 0.8, 0.9, 1.0],
            "colsample_bytree" : [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            "min_child_weight" : [1, 2, 3, 5, 7],
            "gamma"            : [0, 0.05, 0.1, 0.2, 0.3, 0.5],
            "reg_alpha"        : [0, 0.01, 0.05, 0.1, 0.5, 1.0],
            "reg_lambda"       : [0.5, 1.0, 1.5, 2.0, 3.0],
            "scale_pos_weight" : [1.0, 1.5, spw, spw*1.5, spw*2.0],
            "max_delta_step"   : [0, 1, 5],   # helps with imbalanced data
        }
    )

# 4. KNN — uses scaled data, much wider metric/neighbour grid
MODELS["KNN"] = (
    KNeighborsClassifier(),
    {
        "n_neighbors": [15, 21,25],
        "weights"    : ["distance"],
        "metric"     : ["euclidean", "manhattan"],
        "p"          : [1, 2, 3],
        "leaf_size"  : [10, 20, 30, 40, 50],
        "algorithm"  : ["auto", "ball_tree", "kd_tree"],
    }
)

# ── Train + tune + threshold-optimise each model ──────────────────────────────
results = {}
best_rf_model = None

print(f"\n{'Model':<18} {'Best AUC(CV)':>12} {'Thresh':>7} "
      f"{'Test Acc':>9} {'Test AUC':>9} {'Prec':>7} {'Rec':>7} {'F1':>7}")
print("─" * 82)

# ── IMPROVEMENT 6: n_iter=100 (from 60) for more thorough search ──────────────
SEARCH_ITER = 100

for name, (base_model, param_grid) in MODELS.items():
    search = RandomizedSearchCV(
        base_model, param_grid,
        n_iter=SEARCH_ITER, cv=5, scoring="roc_auc",
        n_jobs=-1, refit=True, random_state=SEED
    )
    search.fit(Xs_tr, ys_tr)
    best_model = search.best_estimator_

    # Youden's J threshold on training CV
    thresh = best_threshold_youden(best_model, Xs_tr, ys_tr)

    # Evaluate on real held-out test set (scaled)
    ypr = best_model.predict_proba(X_te_scaled)[:, 1]
    yp  = (ypr >= thresh).astype(int)

    acc  = accuracy_score(y_te, yp)
    auc  = roc_auc_score(y_te, ypr) if len(np.unique(y_te)) > 1 else np.nan
    prec = precision_score(y_te, yp, zero_division=0)
    rec  = recall_score(y_te, yp, zero_division=0)
    f1   = f1_score(y_te, yp, zero_division=0)

    results[name] = {
        "model"       : best_model,
        "best_params" : search.best_params_,
        "cv_auc"      : search.best_score_,
        "threshold"   : thresh,
        "y_pred"      : yp,
        "y_prob"      : ypr,
        "accuracy"    : acc,
        "auc"         : auc,
        "precision"   : prec,
        "recall"      : rec,
        "f1"          : f1,
    }

    print(f"{name:<18} {search.best_score_:>12.3f} {thresh:>7.2f} "
          f"{acc:>9.3f} {auc:>9.3f} {prec:>7.3f} {rec:>7.3f} {f1:>7.3f}")

    if name == "Random Forest":
        best_rf_model = best_model

# ── Detailed reports ──────────────────────────────────────────────────────────
print("\n" + "="*55)
print("DETAILED CLASSIFICATION REPORTS (real held-out test set)")
print("="*55)
for name, r in results.items():
    print(f"\n-- {name} --")
    print(f"  Best params : {r['best_params']}")
    print(f"  Threshold   : {r['threshold']:.2f}  (Youden's J on training CV)")
    print(classification_report(y_te, r["y_pred"], target_names=["No POPF","POPF"]))

best_name = max(results, key=lambda n: results[n]["auc"])
print(f"\n  Best model by Test AUC: {best_name}  (AUC={results[best_name]['auc']:.3f})")

# =============================================================================
# STEP 6 — SHAP + CLINICAL EXPLAINER
# (SHAP fitted on UNscaled X_tr so feature values remain interpretable)
# =============================================================================
CLINICAL_KNOWLEDGE = {
    "hb": {
        "display"    : "Haemoglobin",
        "unit"       : "g/dL",
        "low_thresh" : 10.0, "high_thresh": None,
        "low_status" : "Anaemic", "high_status": "Normal",
        "normal_range": "12-16 g/dL (F), 13-17 g/dL (M)",
        "low_risk_dir": "increases POPF risk", "high_risk_dir": "decreases POPF risk",
        "why_it_matters": (
            "Low haemoglobin indicates pre-operative anaemia and tissue oxygen deprivation. "
            "Poorly oxygenated pancreatic tissue heals slowly, weakening the anastomosis "
            "and making a pancreatic fistula more likely."),
        "doctor_note": "Doctors routinely check Hb but rarely link it to POPF risk.",
    },
    "albumin ": {
        "display"    : "Serum Albumin",
        "unit"       : "g/dL",
        "low_thresh" : 3.5, "high_thresh": None,
        "low_status" : "Hypoalbuminaemia", "high_status": "Normal",
        "normal_range": "3.5-5.0 g/dL",
        "low_risk_dir": "increases POPF risk", "high_risk_dir": "decreases POPF risk",
        "why_it_matters": (
            "Albumin is the primary marker of nutritional status. Hypoalbuminaemia "
            "impairs wound healing, reduces anastomotic tensile strength, and increases "
            "susceptibility to post-operative leaks."),
        "doctor_note": "Serum albumin is checked pre-op but not integrated into POPF risk scoring.",
    },
    "ALP": {
        "display"    : "Alkaline Phosphatase (ALP)",
        "unit"       : "U/L",
        "low_thresh" : None, "high_thresh": 120.0,
        "low_status" : "Normal", "high_status": "Elevated",
        "normal_range": "44-147 U/L",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Elevated ALP reflects biliary obstruction and hepatic stress. Biliary "
            "hypertension raises intra-ductal pancreatic pressure, compromising "
            "anastomotic integrity and increasing fistula risk."),
        "doctor_note": "ALP is ordered as part of a liver panel but not used in POPF prediction.",
    },
    "HBA1C": {
        "display"    : "HbA1c (Glycated Haemoglobin)",
        "unit"       : "%",
        "low_thresh" : None, "high_thresh": 6.5,
        "low_status" : "Normal glycaemic control", "high_status": "Poor glycaemic control",
        "normal_range": "< 5.7% normal, 5.7-6.4% pre-diabetic, >= 6.5% diabetic",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Chronically elevated blood sugar causes microvascular damage and impairs "
            "neutrophil function, delaying anastomotic healing and increasing "
            "vulnerability to post-operative complications."),
        "doctor_note": "HbA1c is checked for diabetics but overlooked as a surgical risk factor.",
    },
    "bill ": {
        "display"    : "Serum Bilirubin",
        "unit"       : "mg/dL",
        "low_thresh" : None, "high_thresh": 2.0,
        "low_status" : "Normal", "high_status": "Hyperbilirubinaemia",
        "normal_range": "0.2-1.2 mg/dL",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Pre-operative jaundice and elevated bilirubin cause Kupffer-cell "
            "dysfunction and endotoxaemia, impairing the immune response and "
            "increasing post-operative fistula complications."),
        "doctor_note": "Bilirubin is well-monitored for biliary disease but not in POPF models.",
    },
    "BLOOD LOSS ": {
        "display"    : "Intra-operative Blood Loss",
        "unit"       : "mL",
        "low_thresh" : None, "high_thresh": 500.0,
        "low_status" : "Acceptable", "high_status": "Excessive",
        "normal_range": "< 500 mL acceptable for Whipple",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Excessive intra-operative haemorrhage triggers coagulopathy and "
            "systemic inflammatory response, impairing anastomotic integrity "
            "and increasing the likelihood of fistula formation."),
        "doctor_note": "Blood loss is recorded but not used quantitatively in POPF prediction.",
    },
    "CBD DIAMETER": {
        "display"    : "Common Bile Duct Diameter",
        "unit"       : "mm",
        "low_thresh" : None, "high_thresh": 8.0,
        "low_status" : "Normal", "high_status": "Dilated",
        "normal_range": "< 6-8 mm",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "A dilated CBD indicates chronic biliary obstruction, leading to hepatic "
            "reserve impairment and elevated intra-ductal pressure, compromising "
            "anastomotic healing after pancreaticoduodenectomy."),
        "doctor_note": "CBD diameter is measured on imaging but rarely used as a POPF predictor.",
    },
    "ca 19-9": {
        "display"    : "CA 19-9 Tumour Marker",
        "unit"       : "U/mL",
        "low_thresh" : None, "high_thresh": 37.0,
        "low_status" : "Normal", "high_status": "Elevated",
        "normal_range": "< 37 U/mL",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Elevated CA 19-9 reflects tumour burden and associated inflammatory "
            "milieu, which may compromise tissue quality and anastomotic healing."),
        "doctor_note": "CA 19-9 is used for diagnosis but not for POPF prediction.",
    },
    "weight loss ": {
        "display"    : "Pre-operative Weight Loss",
        "unit"       : "(binary: 1=yes)",
        "low_thresh" : None, "high_thresh": None,
        "low_status" : "No weight loss", "high_status": "Significant weight loss",
        "normal_range": "Absent",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Pre-operative weight loss is a surrogate for cachexia and protein-energy "
            "malnutrition. Malnourished patients have impaired wound healing and higher "
            "anastomotic complication rates."),
        "doctor_note": "Weight loss is documented clinically but not formally scored for POPF.",
    },
    "fever ": {
        "display"    : "Pre-operative Fever",
        "unit"       : "(binary: 1=yes)",
        "low_thresh" : None, "high_thresh": None,
        "low_status" : "Afebrile", "high_status": "Febrile",
        "normal_range": "Absent",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Pre-operative fever indicates active infection or systemic inflammation. "
            "Operating in an inflammatory state predisposes to impaired anastomotic "
            "healing and higher fistula rates."),
        "doctor_note": "Fever is a general surgical risk factor but not in standard POPF scores.",
    },
}

def get_status(feat, value):
    info = CLINICAL_KNOWLEDGE.get(feat)
    if not info: return "--"
    if info["low_thresh"] and value < info["low_thresh"]:
        return f"{info['low_status']} (< {info['low_thresh']} {info['unit']})"
    if info["high_thresh"] and value > info["high_thresh"]:
        return f"{info['high_status']} (> {info['high_thresh']} {info['unit']})"
    return f"Normal range ({info['normal_range']})"

# Fit SHAP on unscaled training data (interpretable feature values)
print("\n-- Computing SHAP (Random Forest, unscaled X_tr) --")
shap_rf    = best_rf_model
# Re-fit on unscaled data for SHAP interpretability
shap_rf.fit(
    *BorderlineSMOTE(random_state=SEED, k_neighbors=min(4,int(y_tr.sum())-1)).fit_resample(
        X_tr, y_tr)
    if min(4,int(y_tr.sum())-1) >= 1 else (X_tr, y_tr)
)
X_df_shap  = pd.DataFrame(X_tr, columns=final_features)
explainer  = shap.TreeExplainer(shap_rf)
sv         = explainer(X_df_shap)
sv_popf    = sv.values[:, :, 1]


def clinical_explain(idx, X_data, y_data, sv_data, model):
    sv_  = sv_data[idx]
    fv   = X_data[idx]
    prob = model.predict_proba(X_data[idx:idx+1])[0, 1]
    pred = "POPF" if prob >= 0.5 else "No POPF"
    true = "POPF" if y_data[idx] == 1 else "No POPF"

    nt_rows, tr_rows = [], []
    for fi, fname in enumerate(final_features):
        if fname in CLINICAL_KNOWLEDGE or fname in NON_TRADITIONAL:
            nt_rows.append((fi, fname, sv_[fi], fv[fi]))
        else:
            tr_rows.append((fi, fname, sv_[fi], fv[fi]))

    nt_rows.sort(key=lambda x: abs(x[2]), reverse=True)
    tr_rows.sort(key=lambda x: abs(x[2]), reverse=True)

    sep = "=" * 68
    out = (f"\n{sep}\n"
           f"  PATIENT {idx+1}  |  Prediction: {pred}  |  Actual: {true}\n"
           f"  POPF Risk Probability: {prob*100:.1f}%\n"
           f"{sep}\n\n")

    out += "  NON-TRADITIONAL FEATURES\n"
    out += "  " + "-" * 64 + "\n"
    for fi, fname, shap_val, val in nt_rows:
        info    = CLINICAL_KNOWLEDGE.get(fname, {})
        display = info.get("display", NON_TRADITIONAL.get(fname, fname))
        unit    = info.get("unit", "")
        status  = get_status(fname, val)
        arrow   = "increases POPF risk" if shap_val > 0 else "decreases POPF risk"
        out += f"\n  * {display}\n"
        out += f"    Value  : {val:.2f} {unit}\n"
        out += f"    Status : {status}\n"
        out += f"    Effect : {arrow}  (SHAP = {shap_val:+.4f})\n"
        if fname in CLINICAL_KNOWLEDGE:
            why = info["why_it_matters"]
            words = why.split(); line = "      "
            for w in words:
                if len(line) + len(w) > 66: out += line + "\n"; line = "      "
                line += w + " "
            out += line.rstrip() + "\n"

    out += f"\n  TRADITIONAL FEATURES (top 5)\n"
    out += "  " + "-" * 64 + "\n"
    for fi, fname, shap_val, val in tr_rows[:5]:
        arrow = "higher POPF" if shap_val > 0 else "lower POPF"
        out  += f"  . {fname.strip():<25} = {val:>8.2f}   SHAP {shap_val:+.4f}  {arrow}\n"
    return out

popf_idx   = np.where(y_tr == 1)[0][0]
nopopf_idx = np.where(y_tr == 0)[0][0]
print(clinical_explain(popf_idx,   X_tr, y_tr, sv_popf, shap_rf))
print(clinical_explain(nopopf_idx, X_tr, y_tr, sv_popf, shap_rf))

# =============================================================================
# PLOTS (3 figures)
# =============================================================================

# ── Plot 1: YJO Convergence ───────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(10, 4), facecolor=P["bg"])
ax2.set_facecolor(P["card"])
ax2.plot(hist_A, color=P["accent"], lw=2.5, marker="o", ms=5,
         label=f"Stage 1 — all {len(feature_names)} features")
ax2.plot(hist_C, color=P["orange"], lw=2.5, marker="s", ms=5, ls="--",
         label=f"Stage 2 — Set B ({len(set_B)} unimportant features)")
ax2.set_xlabel("Iteration"); ax2.set_ylabel("Best CV AUC")
ax2.set_title("geo Convergence — both stages (AUC fitness)", color=P["text"], fontweight="bold")
ax2.set_ylim(0.4, 1.05)
ax2.legend(facecolor=P["card"], edgecolor=P["border"], fontsize=9)
ax2.grid(True)
plt.tight_layout()
fig2.savefig("D:\\medical_1D\\new_output\\geo_convergence.png",
             dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close(fig2)
print("\nSaved: geo_convergence.png")

# ── Plot 2: Feature Importance ────────────────────────────────────────────────
imp_df = pd.DataFrame({
    "Feature"   : final_features,
    "Importance": shap_rf.feature_importances_,
    "Origin"    : ["A" if f in set_A else "C" for f in final_features],
    "Novel"     : [f in NON_TRADITIONAL for f in final_features],
}).sort_values("Importance", ascending=True)

fig3, ax3 = plt.subplots(figsize=(12, max(7, len(final_features)*0.45 + 2)),
                          facecolor=P["bg"])
ax3.set_facecolor(P["card"])
clrs = []
for _, row in imp_df.iterrows():
    if   row["Novel"] and row["Origin"] == "A": clrs.append(P["green"])
    elif row["Novel"] and row["Origin"] == "C": clrs.append(P["orange"])
    elif row["Origin"] == "A":                  clrs.append(P["blue"])
    else:                                        clrs.append(P["muted"])

bars = ax3.barh(imp_df["Feature"], imp_df["Importance"],
                color=clrs, edgecolor=P["border"], height=0.65, linewidth=0.4)
for bar, (_, row) in zip(bars, imp_df.iterrows()):
    lbl = f'{row["Importance"]:.3f}{"  [NT]" if row["Novel"] else ""}'
    ax3.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
             lbl, va="center", fontsize=8, color=P["muted"])
ax3.set_xlabel("Mean RF Importance (Gini)")
ax3.set_title("CSA-selected Features — importance x origin x novelty",
              color=P["text"], fontweight="bold")
ax3.grid(axis="x")
ax3.legend(handles=[
    mpatches.Patch(color=P["green"],  label="Non-traditional + Set A"),
    mpatches.Patch(color=P["orange"], label="Non-traditional + Set C (rescued)"),
    mpatches.Patch(color=P["blue"],   label="Traditional + Set A"),
    mpatches.Patch(color=P["muted"],  label="Traditional + Set C"),
], facecolor=P["card"], edgecolor=P["border"], fontsize=8, loc="lower right")
plt.tight_layout()
fig3.savefig("D:\\medical_1D\\new_output\\feature_importance_geo.png",
             dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close(fig3)
print("Saved: feature_importance_geo.png")

# ── Plot 3: Multi-Model Comparison ────────────────────────────────────────────
model_names = list(results.keys())
metrics     = ["accuracy", "auc", "precision", "recall", "f1"]
metric_lbls = ["Accuracy", "AUC", "Precision", "Recall", "F1"]
colors_m    = [P["blue"], P["accent"], P["green"], P["orange"], P["yellow"]]
x = np.arange(len(model_names)); w = 0.15

fig5, ax5 = plt.subplots(figsize=(13, 5), facecolor=P["bg"])
ax5.set_facecolor(P["card"])
for mi, (metric, lbl, col) in enumerate(zip(metrics, metric_lbls, colors_m)):
    vals = [results[n][metric] for n in model_names]
    bars = ax5.bar(x + mi*w - 2*w, vals, w, label=lbl,
                   color=col, edgecolor=P["border"], linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f"{v:.2f}", ha="center", va="bottom", fontsize=7,
                 color=P["text"], rotation=90)

thresh_labels = [f"{n}\n(t={results[n]['threshold']:.2f})" for n in model_names]
ax5.set_xticks(x); ax5.set_xticklabels(thresh_labels, fontsize=9)
ax5.set_ylim(0, 1.18); ax5.set_ylabel("Score")
ax5.set_title("Multi-Model Comparison — Real Held-Out Test Set (80/20 split, Youden threshold)",
              color=P["text"], fontweight="bold")
ax5.legend(facecolor=P["card"], edgecolor=P["border"], fontsize=9, loc="upper right")
ax5.grid(axis="y")

best_auc_vals = [results[n]["auc"] for n in model_names]
best_m_idx    = int(np.argmax(best_auc_vals))
ax5.annotate("Best AUC",
             xy=(best_m_idx + 1*w - 2*w, best_auc_vals[best_m_idx]+0.06),
             color=P["yellow"], fontsize=9, fontweight="bold", ha="center")
plt.tight_layout()
fig5.savefig("D:\\medical_1D\\new_output\\model_comparison.png",
             dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close(fig5)
print("Saved: csa_model_comparison.png")

# =============================================================================
# SAVE CLINICAL EXPLANATIONS
# =============================================================================
with open("D:\\medical_1D\\new_output\\clinical_explanations_geo.txt",
          "w", encoding="utf-8") as f:
    f.write("POPF PREDICTION -- CLINICAL EXPLANATIONS\n")
    f.write("Non-Traditional Feature Analysis\n")
    f.write("=" * 68 + "\n\n")
    f.write("GLOBAL FEATURE KNOWLEDGE BASE\n")
    f.write("-" * 68 + "\n\n")
    for fname, info in CLINICAL_KNOWLEDGE.items():
        if fname in final_features:
            fi = final_features.index(fname)
            ms = sv_popf[:, fi].mean()
            f.write(f"FEATURE: {info['display']}\n")
            f.write(f"  Normal range   : {info['normal_range']}\n")
            f.write(f"  Unit           : {info['unit']}\n")
            f.write(f"  Mean SHAP      : {ms:+.4f}\n")
            f.write(f"  Risk direction : {info['high_risk_dir'] if ms>0 else info['low_risk_dir']}\n")
            f.write(f"  Why it matters : {info['why_it_matters']}\n")
            f.write(f"  Doctor note    : {info['doctor_note']}\n\n")

    f.write("\n" + "=" * 68 + "\n")
    f.write("PER-PATIENT EXPLANATIONS (training set)\n")
    f.write("=" * 68 + "\n")
    for idx in range(len(y_tr)):
        f.write(clinical_explain(idx, X_tr, y_tr, sv_popf, shap_rf))

print("Saved: clinical_explanations_geo.txt")
import os
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

out_dir = r"D:\\medical_1D\\new_output"
os.makedirs(out_dir, exist_ok=True)

# ---------------------------------------------------------------------
# 1) CONFUSION MATRIX PLOTS
# ---------------------------------------------------------------------
for name, r in results.items():
    cm = confusion_matrix(y_te, r["y_pred"])

    fig, ax = plt.subplots(figsize=(5.5, 4.5), facecolor=P["bg"])
    ax.set_facecolor(P["card"])

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No POPF", "POPF"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")

    ax.set_title(f"Confusion Matrix — {name}", color=P["text"], fontweight="bold")
    ax.tick_params(colors=P["muted"])
    plt.tight_layout()

    fname = os.path.join(out_dir, f"cm_{name.lower().replace(' ', '_')}.png")
    fig.savefig(fname, dpi=150, bbox_inches="tight", facecolor=P["bg"])
    plt.close(fig)
    print(f"Saved: {fname}")

# ---------------------------------------------------------------------
# 2) FEATURE IMPORTANCE PLOT
#    - tree models: use feature_importances_
#    - non-tree models: fallback to permutation importance
# ---------------------------------------------------------------------
best_model_name = best_name
best_model = results[best_model_name]["model"]

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    importance_source = "Native feature importance"
else:
    print(f"{best_model_name} has no native feature_importances_. Using permutation importance...")
    perm = permutation_importance(
        best_model,
        X_te_scaled,
        y_te,
        n_repeats=30,
        random_state=SEED,
        scoring="roc_auc",
        n_jobs=-1
    )
    importances = perm.importances_mean
    importance_source = "Permutation importance"

imp_df = pd.DataFrame({
    "Feature": final_features,
    "Importance": importances
}).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(12, max(7, len(final_features) * 0.45 + 2)), facecolor=P["bg"])
ax.set_facecolor(P["card"])

bars = ax.barh(
    imp_df["Feature"],
    imp_df["Importance"],
    color=P["accent"],
    edgecolor=P["border"],
    height=0.65
)

for bar, val in zip(bars, imp_df["Importance"]):
    ax.text(
        bar.get_width() + 0.001,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}",
        va="center",
        fontsize=8,
        color=P["muted"]
    )

ax.set_xlabel("Importance")
ax.set_title(f"Feature Importance — {best_model_name} ({importance_source})_geo", color=P["text"], fontweight="bold")
ax.grid(axis="x")
plt.tight_layout()

fname = os.path.join(out_dir, f"feature_importance_{best_model_name.lower().replace(' ', '_')}_geo.png")
fig.savefig(fname, dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close(fig)
print(f"Saved: {fname}")

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "="*55)
print("FINAL SUMMARY")
print(f"  Real train / test split : {len(X_train_r)} / {len(X_test_r)}")
print(f"  Augmented training size : {len(X_aug)}")
print(f"  Final features          : {len(final_features)}")
print(f"\n  -- Test-set results (real held-out patients, Youden threshold) --")
for name, r in results.items():
    mark = "  [BEST AUC]" if name == best_name else ""
    print(f"  {name:<18}  Acc={r['accuracy']:.3f}  AUC={r['auc']:.3f}  "
          f"Rec={r['recall']:.3f}  F1={r['f1']:.3f}  thresh={r['threshold']:.2f}{mark}")
print()

Raw: (76, 54)

── KNN Imputation (k=5) for missing values ──
Binary columns  (mode-imputed) : 7
Continuous cols (KNN-imputed)  : 20

Original: 74 patients | POPF: 43  No-POPF: 31

STEP 2 — 80/20 TRAIN/TEST SPLIT (before augmentation)
Train (real): 59 patients | POPF=34  No-POPF=25
Test  (real): 15  patients | POPF=9  No-POPF=6
  Test set = real patients only, never used during augmentation or tuning

── Linear interpolation augmentation (train only) ──
Real train : 59  |  Augmented train : 300  |  Test (real) : 15
  Binary column integrity verified

STAGE 1 — aco on all features  [n_bees=15, n_iter=20, AUC fitness]
  [GEO Stage-1] Init best=1.0000 features=14
  [GEO Stage-1] Iter 1/20 best=1.0000 features=14
  [GEO Stage-1] Iter 5/20 best=1.0000 features=14
  [GEO Stage-1] Iter 10/20 best=1.0000 features=14
  [GEO Stage-1] Iter 15/20 best=1.0000 features=14
  [GEO Stage-1] Iter 20/20 best=1.0000 features=14
  [GEO Stage-1] Done best=1.0000 features=14

  Set A (important)  : 14  ->  ['

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\medical_1D\\new_output\\geo_convergence.png'

In [3]:
import warnings; warnings.filterwarnings("ignore")
import re, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import shap

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import KNNImputer
from sklearn.model_selection import (StratifiedKFold,
                                     train_test_split, RandomizedSearchCV,
                                     cross_val_predict)
from sklearn.metrics import (classification_report, accuracy_score,
                              roc_auc_score, confusion_matrix,
                              precision_score, recall_score, f1_score)
from imblearn.over_sampling import BorderlineSMOTE
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("⚠ XGBoost not installed — skipping XGBoost model")

SEED = 42
np.random.seed(SEED)

# ── Palette ──────────────────────────────────────────────────────────────────
P = {"bg":"#0F1117","card":"#1A1D27","accent":"#7C6AF7","green":"#4ADE80",
     "red":"#F87171","text":"#E8E8F0","muted":"#8B8FA8","border":"#2A2D3E",
     "orange":"#FB923C","blue":"#60A5FA","yellow":"#FCD34D"}

plt.rcParams.update({
    "figure.facecolor":P["bg"],"axes.facecolor":P["card"],
    "axes.edgecolor":P["border"],"axes.labelcolor":P["text"],
    "xtick.color":P["muted"],"ytick.color":P["muted"],"text.color":P["text"],
    "grid.color":P["border"],"grid.linestyle":"--","grid.alpha":0.4,
    "font.family":"sans-serif","axes.titlesize":11,"axes.labelsize":9,
})

# ── Non-traditional features (paper novelty) ─────────────────────────────────
NON_TRADITIONAL = {
    "hb"             : "Haemoglobin (g/dL)",
    "albumin "       : "Serum Albumin (g/dL)",
    "ALP"            : "Alkaline Phosphatase (U/L)",
    "HBA1C"          : "HbA1c (%)",
    "bill "          : "Serum Bilirubin (mg/dL)",
    "BLOOD LOSS "    : "Intra-op Blood Loss (mL)",
    "CBD DIAMETER"   : "CBD Diameter (mm)",
    "ca 19-9"        : "CA 19-9 (U/mL)",
    "Cea"            : "CEA (ng/mL)",
    "fever "         : "Pre-op Fever",
    "RECENT DIABETES": "Recent Diabetes",
    "comorbidity"    : "Comorbidity",
    "neoadjuvant "   : "Neoadjuvant Treatment",
    "ercp stenting " : "ERCP Stenting",
    "jaundice"       : "Obstructive Jaundice",
    "pain "          : "Pre-op Pain",
    "weight loss "   : "Weight Loss",
    "bleeding "      : "Pre-op Bleeding",
    "Age "           : "Patient Age",
    "Sex"            : "Patient Sex",
}

# =============================================================================
# STEP 1 — LOAD & CLEAN
# =============================================================================
df = pd.read_excel("D:\\medical_1D\\1D data 2.xlsx")
print(f"Raw: {df.shape}")

DROP = ["DGE","PPH","SSI","CLAVIEN DINDO","BILE LAK","HOSPITAL STAY ",
        "READMISION","RESURGERY","mortality",
        "s no ","Name ","Address","CR Number ",
        "Unnamed: 5","Unnamed: 6","Unnamed: 7",
        "date of admisson ","date of sx ","date of discharge ","date of surgery",
        "cect","eus ","periampullary carcinoma ","surgery","CHROMAGANIN "]
df.drop(columns=[c for c in DROP if c in df.columns], inplace=True)

df.dropna(subset=["POPF"], inplace=True)
df["POPF"] = (df["POPF"] != 0).astype(int)

def auto_encode(col, series):
    name = col.strip().lower()
    if series.dtype in [np.float64, np.int64, float, int]:
        num = pd.to_numeric(series, errors="coerce")
        return num if num.notna().mean() >= 0.4 else None
    s = series.astype(str).str.strip()
    if name == "lap open": return None
    if name in ("fever ", "neoadjuvant "):
        def _b(v):
            sv = str(v).strip()
            if sv in ("-","9","nan",""): return np.nan
            try:   return 1.0 if float(sv) > 0 else 0.0
            except: return np.nan
        return series.apply(_b)
    if name == "ercp stenting ":
        return series.apply(lambda v: 1.0 if str(v).strip().startswith("1")
                            else (0.0 if str(v).strip() == "0" else np.nan))
    if name == "comorbidity":
        return series.apply(lambda v: 0.0 if str(v).strip() in ("0","nan","")
                            else (0.0 if str(v).strip() == "0" else 1.0))
    if name == "bill ":
        return pd.to_numeric(s.str.replace(r"[Oo]","0",regex=True), errors="coerce")
    if name == "cea":
        return pd.to_numeric(s.str.replace("`",""), errors="coerce")
    if name == "tumor size":
        return series.apply(lambda v: max(
            (float(n) for n in re.findall(r"[\d.]+", str(v))), default=np.nan))
    num = pd.to_numeric(s.str.replace("`","").str.replace(",","."), errors="coerce")
    return num if num.notna().mean() >= 0.4 else None

y   = df["POPF"].astype(int)
raw = df.drop(columns=["POPF"])
keep = {}
for col in raw.columns:
    res = auto_encode(col, raw[col])
    if res is not None and res.notna().mean() >= 0.20 and res.nunique() > 1:
        keep[col] = res.astype(float)

X_df = pd.DataFrame(keep)
feature_names = list(X_df.columns)

def is_binary_col(series):
    vals = set(series.dropna().unique())
    return vals.issubset({0.0, 1.0})

binary_cols     = [c for c in X_df.columns if is_binary_col(X_df[c])]
continuous_cols = [c for c in X_df.columns if c not in binary_cols]

# ── IMPROVEMENT 1: KNN Imputer for smarter missing value filling ──────────────
print("\n── KNN Imputation (k=5) for missing values ──")
knn_imp = KNNImputer(n_neighbors=5, weights="distance")

# Impute continuous cols with KNN, binary with mode (preserves 0/1 integrity)
X_cont = X_df[continuous_cols].values
X_cont_imputed = knn_imp.fit_transform(X_cont)
for i, col in enumerate(continuous_cols):
    X_df[col] = X_cont_imputed[:, i]
for col in binary_cols:
    X_df[col].fillna(X_df[col].mode()[0], inplace=True)

print(f"Binary columns  (mode-imputed) : {len(binary_cols)}")
print(f"Continuous cols (KNN-imputed)  : {len(continuous_cols)}")

X_np = X_df.values.copy()
y_np = y.values.copy()
binary_idx     = [feature_names.index(c) for c in binary_cols]
continuous_idx = [feature_names.index(c) for c in continuous_cols]
print(f"\nOriginal: {len(y_np)} patients | POPF: {y_np.sum()}  No-POPF: {(y_np==0).sum()}")

# =============================================================================
# STEP 2 — 80/20 TRAIN/TEST SPLIT (BEFORE AUGMENTATION — no leakage)
# =============================================================================
print("\n" + "="*55)
print("STEP 2 — 80/20 TRAIN/TEST SPLIT (before augmentation)")
print("="*55)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_np, y_np, test_size=0.20, stratify=y_np, random_state=SEED)
print(f"Train (real): {len(y_train_r)} patients | POPF={y_train_r.sum()}  No-POPF={(y_train_r==0).sum()}")
print(f"Test  (real): {len(y_test_r)}  patients | POPF={y_test_r.sum()}  No-POPF={(y_test_r==0).sum()}")
print("  Test set = real patients only, never used during augmentation or tuning")

# =============================================================================
# STEP 3 — BINARY-SAFE INTERPOLATION AUGMENTATION (training set only)
# ── IMPROVEMENT 2: TARGET_N raised 200→300 for more robust training ──────────
# =============================================================================
print("\n── Linear interpolation augmentation (train only) ──")

TARGET_N = 300                      # ← raised from 200
cont_idx = [i for i in range(X_np.shape[1]) if i not in binary_idx]
rng_aug  = np.random.default_rng(SEED)

def linear_interpolate_class(X_class, n_needed, binary_idx, cont_idx, rng):
    n_real, n_feat = X_class.shape
    X_syn = np.empty((n_needed, n_feat), dtype=float)
    for s in range(n_needed):
        a, b = rng.choice(n_real, size=2, replace=(n_real < 2))
        lam  = rng.uniform(0.0, 1.0)
        X_syn[s, cont_idx]   = lam * X_class[a, cont_idx] + (1.0-lam) * X_class[b, cont_idx]
        X_syn[s, binary_idx] = np.where(lam >= 0.5,
                                         X_class[a, binary_idx],
                                         X_class[b, binary_idx])
    return X_syn

idx_pos = np.where(y_train_r == 1)[0]
idx_neg = np.where(y_train_r == 0)[0]
X_pos, X_neg = X_train_r[idx_pos], X_train_r[idx_neg]
half      = TARGET_N // 2
n_syn_pos = max(0, half - len(X_pos))
n_syn_neg = max(0, half - len(X_neg))
X_syn_pos = linear_interpolate_class(X_pos, n_syn_pos, binary_idx, cont_idx, rng_aug)
X_syn_neg = linear_interpolate_class(X_neg, n_syn_neg, binary_idx, cont_idx, rng_aug)

X_aug = np.vstack([X_train_r, X_syn_pos, X_syn_neg])
y_aug = np.concatenate([y_train_r,
                        np.ones(n_syn_pos, dtype=int),
                        np.zeros(n_syn_neg, dtype=int)])

print(f"Real train : {len(X_train_r)}  |  Augmented train : {len(X_aug)}  |  Test (real) : {len(X_test_r)}")

for idx in binary_idx:
    vals = set(np.unique(X_aug[:, idx]))
    assert vals.issubset({0.0, 1.0}), f"Binary violated: {feature_names[idx]}"
print("  Binary column integrity verified")

# =============================================================================
# STEP 4 — YJO FEATURE SELECTION (on augmented training set)
# ── IMPROVEMENT 3: AUC-based fitness, more wasps, more iterations ─────────────
# =============================================================================
def yellow_jacket_optimizer(X_data, y_data, n_wasps, n_iter,
                             p_attack_start=0.20, p_attack_end=0.75,
                             n_flip=2, min_feat=3, label="YJO"):
    n_feat = X_data.shape[1]
    rng    = np.random.default_rng(SEED)

    def fitness(mask):
        if mask.sum() == 0: return 0.0
        # ── AUC-based fitness (more discriminative than accuracy) ─────────────
        skf  = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        aucs = []
        for tr, te in skf.split(X_data[:, mask], y_data):
            Xtr, Xte = X_data[tr][:, mask].copy(), X_data[te][:, mask].copy()
            ytr, yte = y_data[tr], y_data[te]
            k = min(4, int(ytr.sum()) - 1)
            if k < 1: continue
            try:
                Xs, ys = BorderlineSMOTE(random_state=SEED, k_neighbors=k).fit_resample(Xtr, ytr)
            except Exception:
                Xs, ys = Xtr.copy(), ytr.copy()
            rf = RandomForestClassifier(n_estimators=50, max_depth=5,
                                        class_weight="balanced", random_state=SEED)
            rf.fit(Xs, ys)
            prob = rf.predict_proba(Xte)[:, 1]
            if len(np.unique(yte)) > 1:
                aucs.append(roc_auc_score(yte, prob))
        return float(np.mean(aucs)) if aucs else 0.0

    def enforce_min(mask):
        m = mask.copy()
        if m.sum() < min_feat:
            off = np.where(m == 0)[0]
            needed = min_feat - int(m.sum())
            m[rng.choice(off, min(needed, len(off)), replace=False)] = 1.0
        return m

    def random_wasp():
        m = rng.integers(0, 2, n_feat).astype(float)
        return enforce_min(m)

    colony  = np.array([random_wasp() for _ in range(n_wasps)])
    scores  = np.array([fitness(colony[i].astype(bool)) for i in range(n_wasps)])
    nests, nest_scores = colony.copy(), scores.copy()
    g_idx   = int(np.argmax(scores))
    g_best  = colony[g_idx].copy()
    g_score = float(scores[g_idx])
    history = [g_score]
    print(f"  [{label}] Init  best={g_score:.4f}  features={int(g_best.sum())}")

    for it in range(n_iter):
        t        = it / max(n_iter - 1, 1)
        p_attack = p_attack_start + t * (p_attack_end - p_attack_start)
        for i in range(n_wasps):
            attack_bits = rng.random(n_feat) < p_attack
            new_mask    = np.where(attack_bits, g_best, colony[i]).copy()
            flip_idx    = rng.choice(n_feat, min(n_flip, n_feat), replace=False)
            new_mask[flip_idx] = 1.0 - new_mask[flip_idx]
            new_mask = enforce_min(new_mask)
            sc = fitness(new_mask.astype(bool))
            colony[i] = new_mask; scores[i] = sc
            if sc > nest_scores[i]: nests[i] = new_mask.copy(); nest_scores[i] = sc
            if sc > g_score: g_best = new_mask.copy(); g_score = sc
        worst_i = int(np.argmin(scores))
        colony[worst_i] = random_wasp()
        scores[worst_i] = fitness(colony[worst_i].astype(bool))
        if scores[worst_i] > g_score: g_best = colony[worst_i].copy(); g_score = scores[worst_i]
        history.append(g_score)
        if (it + 1) % 5 == 0 or it == 0:
            print(f"  [{label}] Iter {it+1:2d}/{n_iter}  p_attack={p_attack:.2f}  "
                  f"best={g_score:.4f}  features={int(g_best.sum())}")
    print(f"  [{label}] Done  best={g_score:.4f}  features={int(g_best.sum())}")
    return g_best.astype(bool), g_score, history

print("\n" + "="*55)
print("STAGE 1 — YJO on all features  [n_wasps=15, n_iter=20, AUC fitness]")
print("="*55)
mask_A, score_A, hist_A = yellow_jacket_optimizer(
    X_aug, y_aug, n_wasps=15, n_iter=20,          # ← raised from 10/12
    p_attack_start=0.20, p_attack_end=0.75, n_flip=2, min_feat=4, label="Stage-1")
set_A = [feature_names[i] for i, m in enumerate(mask_A) if m]
set_B = [feature_names[i] for i, m in enumerate(mask_A) if not m]
print(f"\n  Set A (important)  : {len(set_A)}  ->  {set_A}")
print(f"  Set B (unimportant): {len(set_B)}  ->  {set_B}")

print("\n" + "="*55)
print("STAGE 2 — YJO on unimportant Set B  [n_wasps=12, n_iter=15]")
print("="*55)
idx_B  = [i for i, m in enumerate(mask_A) if not m]
mask_C, score_C, hist_C = yellow_jacket_optimizer(
    X_aug[:, idx_B], y_aug, n_wasps=12, n_iter=15,  # ← raised from 8/10
    p_attack_start=0.15, p_attack_end=0.70, n_flip=3, min_feat=2, label="Stage-2")
set_C = [set_B[i] for i, m in enumerate(mask_C) if m]
print(f"\n  Set C (rescued): {len(set_C)}  ->  {set_C}")

final_features = list(dict.fromkeys(set_A + set_C))
print(f"\nFinal features (A + C): {len(final_features)}")
for f in final_features:
    tag = "  [NON-TRAD]" if f in NON_TRADITIONAL else ""
    src = "A" if f in set_A else "C"
    print(f"  [{src}]  {f}{tag}")

# =============================================================================
# STEP 5 — MULTI-MODEL TRAINING + DEEP HYPERPARAMETER TUNING
# ── IMPROVEMENT 4: RobustScaler pipelines, BorderlineSMOTE, Youden threshold,
#                   n_iter=100, wider grids, RepeatedStratifiedKFold ──────────
# =============================================================================
feat_idx = [feature_names.index(f) for f in final_features]
X_tr     = X_aug[:, feat_idx].copy()
y_tr     = y_aug.copy()
X_te     = X_test_r[:, feat_idx].copy()
y_te     = y_test_r.copy()

# ── Feature scaling on continuous columns only ────────────────────────────────
final_cont_mask = np.array([f not in binary_cols for f in final_features])
scaler = RobustScaler()

def scale_data(X_train, X_test, cont_mask):
    """Fit scaler on train continuous cols, transform both — no leakage."""
    X_tr_s = X_train.copy()
    X_te_s = X_test.copy()
    if cont_mask.any():
        X_tr_s[:, cont_mask] = scaler.fit_transform(X_train[:, cont_mask])
        X_te_s[:, cont_mask] = scaler.transform(X_test[:, cont_mask])
    return X_tr_s, X_te_s

X_tr_scaled, X_te_scaled = scale_data(X_tr, X_te, final_cont_mask)

# ── BorderlineSMOTE on augmented training set ─────────────────────────────────
k_sm = min(4, int(y_tr.sum()) - 1)
try:
    Xs_tr, ys_tr = BorderlineSMOTE(random_state=SEED, k_neighbors=k_sm).fit_resample(
        X_tr_scaled, y_tr)
except Exception:
    from imblearn.over_sampling import SMOTE
    Xs_tr, ys_tr = SMOTE(random_state=SEED, k_neighbors=k_sm).fit_resample(
        X_tr_scaled, y_tr)

neg_count = int((ys_tr == 0).sum())
pos_count = int((ys_tr == 1).sum())
spw       = round(neg_count / max(pos_count, 1), 2)
print(f"\nBorderlineSMOTE output -> neg={neg_count}  pos={pos_count}  scale_pos_weight={spw}")

print("\n" + "="*55)
print("STEP 5 — MULTI-MODEL TRAINING + DEEP HYPERPARAMETER TUNING")
print(f"  Train : {len(Xs_tr)} (BorderlineSMOTE on augmented) | Test : {len(X_te_scaled)} (real)")
print(f"  Scaling: RobustScaler on {final_cont_mask.sum()} continuous features")
print("="*55)

# ── IMPROVEMENT 5: Youden's J threshold (sensitivity + specificity − 1) ──────
def best_threshold_youden(model, X, y):
    """
    Find threshold maximising Youden's J = Sensitivity + Specificity − 1.
    More robust than F1-only for imbalanced medical data.
    Note: cross_val_predict requires partitions so we use StratifiedKFold(10).
    """
    skf      = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
    cv_prob  = cross_val_predict(model, X, y, cv=skf, method="predict_proba")[:, 1]
    thresholds = np.arange(0.20, 0.81, 0.02)
    best_t, best_j = 0.5, -1.0
    for t in thresholds:
        yp  = (cv_prob >= t).astype(int)
        rec = recall_score(y, yp, zero_division=0)                    # sensitivity
        tn  = confusion_matrix(y, yp).ravel()
        # specificity = TN / (TN + FP)
        cm  = confusion_matrix(y, yp)
        if cm.shape == (2, 2):
            spec = cm[0, 0] / max(cm[0, 0] + cm[0, 1], 1)
        else:
            spec = 0.0
        j = rec + spec - 1.0
        if j > best_j:
            best_j = j
            best_t = t
    return float(best_t)

# ── Model definitions ─────────────────────────────────────────────────────────
MODELS = {}

# 1. Random Forest — wider grid, more estimators
MODELS["Random Forest"] = (
    RandomForestClassifier(random_state=SEED),
    {
        "n_estimators": [500, 700, 1000],
        "max_depth": [6, 8, 10, 12, None],
        "min_samples_leaf"  : [1, 2],
        "min_samples_split" : [2,3],
        "max_features"      : ["sqrt",0.7,None],
        "class_weight"      : ["balanced", "balanced_subsample",
                               {0:1,1:2}, {0:1,1:3}, {0:1,1:4}],
        "max_samples"       : [0.7, 0.8, 0.9, None],
    }
)

# 2. Decision Tree
MODELS["Decision Tree"] = (
    DecisionTreeClassifier(random_state=SEED),
    {
        "max_depth"         : [3, 4, 5, 6, 7, 8, None],
        "min_samples_leaf"  : [1, 2, 3, 5],
        "min_samples_split" : [2, 4, 6],
        "criterion"         : ["gini", "entropy"],
        "class_weight"      : ["balanced", {0:1,1:2}, {0:1,1:3}, {0:1,1:4}],
        "ccp_alpha"         : [0.0, 0.002, 0.005, 0.01, 0.02, 0.05],
        "splitter"          : ["best", "random"],
    }
)

# 3. XGBoost — deeper regularisation grid
if HAS_XGB:
    MODELS["XGBoost"] = (
        XGBClassifier(random_state=SEED, eval_metric="logloss", verbosity=0),
        {
            "n_estimators"     : [100, 200, 300, 500, 700],
            "max_depth"        : [3, 4, 5, 6, 7],
            "learning_rate"    : [0.005, 0.01, 0.05, 0.1, 0.2],
            "subsample"        : [0.6, 0.7, 0.8, 0.9, 1.0],
            "colsample_bytree" : [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            "min_child_weight" : [1, 2, 3, 5, 7],
            "gamma"            : [0, 0.05, 0.1, 0.2, 0.3, 0.5],
            "reg_alpha"        : [0, 0.01, 0.05, 0.1, 0.5, 1.0],
            "reg_lambda"       : [0.5, 1.0, 1.5, 2.0, 3.0],
            "scale_pos_weight" : [1.0, 1.5, spw, spw*1.5, spw*2.0],
            "max_delta_step"   : [0, 1, 5],   # helps with imbalanced data
        }
    )

# 4. KNN — uses scaled data, much wider metric/neighbour grid
MODELS["KNN"] = (
    KNeighborsClassifier(),
    {
        "n_neighbors": [15, 21,25],
        "weights"    : ["distance"],
        "metric"     : ["euclidean", "manhattan"],
        "p"          : [1, 2, 3],
        "leaf_size"  : [10, 20, 30, 40, 50],
        "algorithm"  : ["auto", "ball_tree", "kd_tree"],
    }
)

# ── Train + tune + threshold-optimise each model ──────────────────────────────
results = {}
best_rf_model = None

print(f"\n{'Model':<18} {'Best AUC(CV)':>12} {'Thresh':>7} "
      f"{'Test Acc':>9} {'Test AUC':>9} {'Prec':>7} {'Rec':>7} {'F1':>7}")
print("─" * 82)

# ── IMPROVEMENT 6: n_iter=100 (from 60) for more thorough search ──────────────
SEARCH_ITER = 100

for name, (base_model, param_grid) in MODELS.items():
    search = RandomizedSearchCV(
        base_model, param_grid,
        n_iter=SEARCH_ITER, cv=5, scoring="roc_auc",
        n_jobs=-1, refit=True, random_state=SEED
    )
    search.fit(Xs_tr, ys_tr)
    best_model = search.best_estimator_

    # Youden's J threshold on training CV
    thresh = best_threshold_youden(best_model, Xs_tr, ys_tr)

    # Evaluate on real held-out test set (scaled)
    ypr = best_model.predict_proba(X_te_scaled)[:, 1]
    yp  = (ypr >= thresh).astype(int)

    acc  = accuracy_score(y_te, yp)
    auc  = roc_auc_score(y_te, ypr) if len(np.unique(y_te)) > 1 else np.nan
    prec = precision_score(y_te, yp, zero_division=0)
    rec  = recall_score(y_te, yp, zero_division=0)
    f1   = f1_score(y_te, yp, zero_division=0)

    results[name] = {
        "model"       : best_model,
        "best_params" : search.best_params_,
        "cv_auc"      : search.best_score_,
        "threshold"   : thresh,
        "y_pred"      : yp,
        "y_prob"      : ypr,
        "accuracy"    : acc,
        "auc"         : auc,
        "precision"   : prec,
        "recall"      : rec,
        "f1"          : f1,
    }

    print(f"{name:<18} {search.best_score_:>12.3f} {thresh:>7.2f} "
          f"{acc:>9.3f} {auc:>9.3f} {prec:>7.3f} {rec:>7.3f} {f1:>7.3f}")

    if name == "Random Forest":
        best_rf_model = best_model

# ── Detailed reports ──────────────────────────────────────────────────────────
print("\n" + "="*55)
print("DETAILED CLASSIFICATION REPORTS (real held-out test set)")
print("="*55)
for name, r in results.items():
    print(f"\n-- {name} --")
    print(f"  Best params : {r['best_params']}")
    print(f"  Threshold   : {r['threshold']:.2f}  (Youden's J on training CV)")
    print(classification_report(y_te, r["y_pred"], target_names=["No POPF","POPF"]))

best_name = max(results, key=lambda n: results[n]["auc"])
print(f"\n  Best model by Test AUC: {best_name}  (AUC={results[best_name]['auc']:.3f})")

# =============================================================================
# STEP 6 — SHAP + CLINICAL EXPLAINER
# (SHAP fitted on UNscaled X_tr so feature values remain interpretable)
# =============================================================================
CLINICAL_KNOWLEDGE = {
    "hb": {
        "display"    : "Haemoglobin",
        "unit"       : "g/dL",
        "low_thresh" : 10.0, "high_thresh": None,
        "low_status" : "Anaemic", "high_status": "Normal",
        "normal_range": "12-16 g/dL (F), 13-17 g/dL (M)",
        "low_risk_dir": "increases POPF risk", "high_risk_dir": "decreases POPF risk",
        "why_it_matters": (
            "Low haemoglobin indicates pre-operative anaemia and tissue oxygen deprivation. "
            "Poorly oxygenated pancreatic tissue heals slowly, weakening the anastomosis "
            "and making a pancreatic fistula more likely."),
        "doctor_note": "Doctors routinely check Hb but rarely link it to POPF risk.",
    },
    "albumin ": {
        "display"    : "Serum Albumin",
        "unit"       : "g/dL",
        "low_thresh" : 3.5, "high_thresh": None,
        "low_status" : "Hypoalbuminaemia", "high_status": "Normal",
        "normal_range": "3.5-5.0 g/dL",
        "low_risk_dir": "increases POPF risk", "high_risk_dir": "decreases POPF risk",
        "why_it_matters": (
            "Albumin is the primary marker of nutritional status. Hypoalbuminaemia "
            "impairs wound healing, reduces anastomotic tensile strength, and increases "
            "susceptibility to post-operative leaks."),
        "doctor_note": "Serum albumin is checked pre-op but not integrated into POPF risk scoring.",
    },
    "ALP": {
        "display"    : "Alkaline Phosphatase (ALP)",
        "unit"       : "U/L",
        "low_thresh" : None, "high_thresh": 120.0,
        "low_status" : "Normal", "high_status": "Elevated",
        "normal_range": "44-147 U/L",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Elevated ALP reflects biliary obstruction and hepatic stress. Biliary "
            "hypertension raises intra-ductal pancreatic pressure, compromising "
            "anastomotic integrity and increasing fistula risk."),
        "doctor_note": "ALP is ordered as part of a liver panel but not used in POPF prediction.",
    },
    "HBA1C": {
        "display"    : "HbA1c (Glycated Haemoglobin)",
        "unit"       : "%",
        "low_thresh" : None, "high_thresh": 6.5,
        "low_status" : "Normal glycaemic control", "high_status": "Poor glycaemic control",
        "normal_range": "< 5.7% normal, 5.7-6.4% pre-diabetic, >= 6.5% diabetic",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Chronically elevated blood sugar causes microvascular damage and impairs "
            "neutrophil function, delaying anastomotic healing and increasing "
            "vulnerability to post-operative complications."),
        "doctor_note": "HbA1c is checked for diabetics but overlooked as a surgical risk factor.",
    },
    "bill ": {
        "display"    : "Serum Bilirubin",
        "unit"       : "mg/dL",
        "low_thresh" : None, "high_thresh": 2.0,
        "low_status" : "Normal", "high_status": "Hyperbilirubinaemia",
        "normal_range": "0.2-1.2 mg/dL",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Pre-operative jaundice and elevated bilirubin cause Kupffer-cell "
            "dysfunction and endotoxaemia, impairing the immune response and "
            "increasing post-operative fistula complications."),
        "doctor_note": "Bilirubin is well-monitored for biliary disease but not in POPF models.",
    },
    "BLOOD LOSS ": {
        "display"    : "Intra-operative Blood Loss",
        "unit"       : "mL",
        "low_thresh" : None, "high_thresh": 500.0,
        "low_status" : "Acceptable", "high_status": "Excessive",
        "normal_range": "< 500 mL acceptable for Whipple",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Excessive intra-operative haemorrhage triggers coagulopathy and "
            "systemic inflammatory response, impairing anastomotic integrity "
            "and increasing the likelihood of fistula formation."),
        "doctor_note": "Blood loss is recorded but not used quantitatively in POPF prediction.",
    },
    "CBD DIAMETER": {
        "display"    : "Common Bile Duct Diameter",
        "unit"       : "mm",
        "low_thresh" : None, "high_thresh": 8.0,
        "low_status" : "Normal", "high_status": "Dilated",
        "normal_range": "< 6-8 mm",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "A dilated CBD indicates chronic biliary obstruction, leading to hepatic "
            "reserve impairment and elevated intra-ductal pressure, compromising "
            "anastomotic healing after pancreaticoduodenectomy."),
        "doctor_note": "CBD diameter is measured on imaging but rarely used as a POPF predictor.",
    },
    "ca 19-9": {
        "display"    : "CA 19-9 Tumour Marker",
        "unit"       : "U/mL",
        "low_thresh" : None, "high_thresh": 37.0,
        "low_status" : "Normal", "high_status": "Elevated",
        "normal_range": "< 37 U/mL",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Elevated CA 19-9 reflects tumour burden and associated inflammatory "
            "milieu, which may compromise tissue quality and anastomotic healing."),
        "doctor_note": "CA 19-9 is used for diagnosis but not for POPF prediction.",
    },
    "weight loss ": {
        "display"    : "Pre-operative Weight Loss",
        "unit"       : "(binary: 1=yes)",
        "low_thresh" : None, "high_thresh": None,
        "low_status" : "No weight loss", "high_status": "Significant weight loss",
        "normal_range": "Absent",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Pre-operative weight loss is a surrogate for cachexia and protein-energy "
            "malnutrition. Malnourished patients have impaired wound healing and higher "
            "anastomotic complication rates."),
        "doctor_note": "Weight loss is documented clinically but not formally scored for POPF.",
    },
    "fever ": {
        "display"    : "Pre-operative Fever",
        "unit"       : "(binary: 1=yes)",
        "low_thresh" : None, "high_thresh": None,
        "low_status" : "Afebrile", "high_status": "Febrile",
        "normal_range": "Absent",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Pre-operative fever indicates active infection or systemic inflammation. "
            "Operating in an inflammatory state predisposes to impaired anastomotic "
            "healing and higher fistula rates."),
        "doctor_note": "Fever is a general surgical risk factor but not in standard POPF scores.",
    },
}

def get_status(feat, value):
    info = CLINICAL_KNOWLEDGE.get(feat)
    if not info: return "--"
    if info["low_thresh"] and value < info["low_thresh"]:
        return f"{info['low_status']} (< {info['low_thresh']} {info['unit']})"
    if info["high_thresh"] and value > info["high_thresh"]:
        return f"{info['high_status']} (> {info['high_thresh']} {info['unit']})"
    return f"Normal range ({info['normal_range']})"

# Fit SHAP on unscaled training data (interpretable feature values)
print("\n-- Computing SHAP (Random Forest, unscaled X_tr) --")
shap_rf    = best_rf_model
# Re-fit on unscaled data for SHAP interpretability
shap_rf.fit(
    *BorderlineSMOTE(random_state=SEED, k_neighbors=min(4,int(y_tr.sum())-1)).fit_resample(
        X_tr, y_tr)
    if min(4,int(y_tr.sum())-1) >= 1 else (X_tr, y_tr)
)
X_df_shap  = pd.DataFrame(X_tr, columns=final_features)
explainer  = shap.TreeExplainer(shap_rf)
sv         = explainer(X_df_shap)
sv_popf    = sv.values[:, :, 1]


def clinical_explain(idx, X_data, y_data, sv_data, model):
    sv_  = sv_data[idx]
    fv   = X_data[idx]
    prob = model.predict_proba(X_data[idx:idx+1])[0, 1]
    pred = "POPF" if prob >= 0.5 else "No POPF"
    true = "POPF" if y_data[idx] == 1 else "No POPF"

    nt_rows, tr_rows = [], []
    for fi, fname in enumerate(final_features):
        if fname in CLINICAL_KNOWLEDGE or fname in NON_TRADITIONAL:
            nt_rows.append((fi, fname, sv_[fi], fv[fi]))
        else:
            tr_rows.append((fi, fname, sv_[fi], fv[fi]))

    nt_rows.sort(key=lambda x: abs(x[2]), reverse=True)
    tr_rows.sort(key=lambda x: abs(x[2]), reverse=True)

    sep = "=" * 68
    out = (f"\n{sep}\n"
           f"  PATIENT {idx+1}  |  Prediction: {pred}  |  Actual: {true}\n"
           f"  POPF Risk Probability: {prob*100:.1f}%\n"
           f"{sep}\n\n")

    out += "  NON-TRADITIONAL FEATURES\n"
    out += "  " + "-" * 64 + "\n"
    for fi, fname, shap_val, val in nt_rows:
        info    = CLINICAL_KNOWLEDGE.get(fname, {})
        display = info.get("display", NON_TRADITIONAL.get(fname, fname))
        unit    = info.get("unit", "")
        status  = get_status(fname, val)
        arrow   = "increases POPF risk" if shap_val > 0 else "decreases POPF risk"
        out += f"\n  * {display}\n"
        out += f"    Value  : {val:.2f} {unit}\n"
        out += f"    Status : {status}\n"
        out += f"    Effect : {arrow}  (SHAP = {shap_val:+.4f})\n"
        if fname in CLINICAL_KNOWLEDGE:
            why = info["why_it_matters"]
            words = why.split(); line = "      "
            for w in words:
                if len(line) + len(w) > 66: out += line + "\n"; line = "      "
                line += w + " "
            out += line.rstrip() + "\n"

    out += f"\n  TRADITIONAL FEATURES (top 5)\n"
    out += "  " + "-" * 64 + "\n"
    for fi, fname, shap_val, val in tr_rows[:5]:
        arrow = "higher POPF" if shap_val > 0 else "lower POPF"
        out  += f"  . {fname.strip():<25} = {val:>8.2f}   SHAP {shap_val:+.4f}  {arrow}\n"
    return out

popf_idx   = np.where(y_tr == 1)[0][0]
nopopf_idx = np.where(y_tr == 0)[0][0]
print(clinical_explain(popf_idx,   X_tr, y_tr, sv_popf, shap_rf))
print(clinical_explain(nopopf_idx, X_tr, y_tr, sv_popf, shap_rf))

# =============================================================================
# PLOTS (3 figures)
# =============================================================================

# ── Plot 1: YJO Convergence ───────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(10, 4), facecolor=P["bg"])
ax2.set_facecolor(P["card"])
ax2.plot(hist_A, color=P["accent"], lw=2.5, marker="o", ms=5,
         label=f"Stage 1 — all {len(feature_names)} features")
ax2.plot(hist_C, color=P["orange"], lw=2.5, marker="s", ms=5, ls="--",
         label=f"Stage 2 — Set B ({len(set_B)} unimportant features)")
ax2.set_xlabel("Iteration"); ax2.set_ylabel("Best CV AUC")
ax2.set_title("YJO Convergence — both stages (AUC fitness)", color=P["text"], fontweight="bold")
ax2.set_ylim(0.4, 1.05)
ax2.legend(facecolor=P["card"], edgecolor=P["border"], fontsize=9)
ax2.grid(True)
plt.tight_layout()
fig2.savefig("D:\\medical_1D\\new_output\\yjo_convergence.png",
             dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close(fig2)
print("\nSaved: yjo_convergence.png")

# ── Plot 2: Feature Importance ────────────────────────────────────────────────
imp_df = pd.DataFrame({
    "Feature"   : final_features,
    "Importance": shap_rf.feature_importances_,
    "Origin"    : ["A" if f in set_A else "C" for f in final_features],
    "Novel"     : [f in NON_TRADITIONAL for f in final_features],
}).sort_values("Importance", ascending=True)

fig3, ax3 = plt.subplots(figsize=(12, max(7, len(final_features)*0.45 + 2)),
                          facecolor=P["bg"])
ax3.set_facecolor(P["card"])
clrs = []
for _, row in imp_df.iterrows():
    if   row["Novel"] and row["Origin"] == "A": clrs.append(P["green"])
    elif row["Novel"] and row["Origin"] == "C": clrs.append(P["orange"])
    elif row["Origin"] == "A":                  clrs.append(P["blue"])
    else:                                        clrs.append(P["muted"])

bars = ax3.barh(imp_df["Feature"], imp_df["Importance"],
                color=clrs, edgecolor=P["border"], height=0.65, linewidth=0.4)
for bar, (_, row) in zip(bars, imp_df.iterrows()):
    lbl = f'{row["Importance"]:.3f}{"  [NT]" if row["Novel"] else ""}'
    ax3.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
             lbl, va="center", fontsize=8, color=P["muted"])
ax3.set_xlabel("Mean RF Importance (Gini)")
ax3.set_title("YJO-selected Features — importance x origin x novelty",
              color=P["text"], fontweight="bold")
ax3.grid(axis="x")
ax3.legend(handles=[
    mpatches.Patch(color=P["green"],  label="Non-traditional + Set A"),
    mpatches.Patch(color=P["orange"], label="Non-traditional + Set C (rescued)"),
    mpatches.Patch(color=P["blue"],   label="Traditional + Set A"),
    mpatches.Patch(color=P["muted"],  label="Traditional + Set C"),
], facecolor=P["card"], edgecolor=P["border"], fontsize=8, loc="lower right")
plt.tight_layout()
fig3.savefig("D:\\medical_1D\\new_output\\feature_importance_yjo.png",
             dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close(fig3)
print("Saved: feature_importance_yjo.png")

# ── Plot 3: Multi-Model Comparison ────────────────────────────────────────────
model_names = list(results.keys())
metrics     = ["accuracy", "auc", "precision", "recall", "f1"]
metric_lbls = ["Accuracy", "AUC", "Precision", "Recall", "F1"]
colors_m    = [P["blue"], P["accent"], P["green"], P["orange"], P["yellow"]]
x = np.arange(len(model_names)); w = 0.15

fig5, ax5 = plt.subplots(figsize=(13, 5), facecolor=P["bg"])
ax5.set_facecolor(P["card"])
for mi, (metric, lbl, col) in enumerate(zip(metrics, metric_lbls, colors_m)):
    vals = [results[n][metric] for n in model_names]
    bars = ax5.bar(x + mi*w - 2*w, vals, w, label=lbl,
                   color=col, edgecolor=P["border"], linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f"{v:.2f}", ha="center", va="bottom", fontsize=7,
                 color=P["text"], rotation=90)

thresh_labels = [f"{n}\n(t={results[n]['threshold']:.2f})" for n in model_names]
ax5.set_xticks(x); ax5.set_xticklabels(thresh_labels, fontsize=9)
ax5.set_ylim(0, 1.18); ax5.set_ylabel("Score")
ax5.set_title("Multi-Model Comparison — Real Held-Out Test Set (80/20 split, Youden threshold)",
              color=P["text"], fontweight="bold")
ax5.legend(facecolor=P["card"], edgecolor=P["border"], fontsize=9, loc="upper right")
ax5.grid(axis="y")

best_auc_vals = [results[n]["auc"] for n in model_names]
best_m_idx    = int(np.argmax(best_auc_vals))
ax5.annotate("Best AUC",
             xy=(best_m_idx + 1*w - 2*w, best_auc_vals[best_m_idx]+0.06),
             color=P["yellow"], fontsize=9, fontweight="bold", ha="center")
plt.tight_layout()
fig5.savefig("D:\\medical_1D\\new_output\\model_comparison.png",
             dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close(fig5)
print("Saved: model_comparison.png")

# =============================================================================
# SAVE CLINICAL EXPLANATIONS
# =============================================================================
with open("D:\\medical_1D\\new_output\\clinical_explanations.txt",
          "w", encoding="utf-8") as f:
    f.write("POPF PREDICTION -- CLINICAL EXPLANATIONS\n")
    f.write("Non-Traditional Feature Analysis\n")
    f.write("=" * 68 + "\n\n")
    f.write("GLOBAL FEATURE KNOWLEDGE BASE\n")
    f.write("-" * 68 + "\n\n")
    for fname, info in CLINICAL_KNOWLEDGE.items():
        if fname in final_features:
            fi = final_features.index(fname)
            ms = sv_popf[:, fi].mean()
            f.write(f"FEATURE: {info['display']}\n")
            f.write(f"  Normal range   : {info['normal_range']}\n")
            f.write(f"  Unit           : {info['unit']}\n")
            f.write(f"  Mean SHAP      : {ms:+.4f}\n")
            f.write(f"  Risk direction : {info['high_risk_dir'] if ms>0 else info['low_risk_dir']}\n")
            f.write(f"  Why it matters : {info['why_it_matters']}\n")
            f.write(f"  Doctor note    : {info['doctor_note']}\n\n")

    f.write("\n" + "=" * 68 + "\n")
    f.write("PER-PATIENT EXPLANATIONS (training set)\n")
    f.write("=" * 68 + "\n")
    for idx in range(len(y_tr)):
        f.write(clinical_explain(idx, X_tr, y_tr, sv_popf, shap_rf))

print("Saved: clinical_explanations.txt")
import os
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

out_dir = r"D:\\medical_1D\\new_output"
os.makedirs(out_dir, exist_ok=True)

# ---------------------------------------------------------------------
# 1) CONFUSION MATRIX PLOTS
# ---------------------------------------------------------------------
for name, r in results.items():
    cm = confusion_matrix(y_te, r["y_pred"])

    fig, ax = plt.subplots(figsize=(5.5, 4.5), facecolor=P["bg"])
    ax.set_facecolor(P["card"])

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No POPF", "POPF"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")

    ax.set_title(f"Confusion Matrix — {name}", color=P["text"], fontweight="bold")
    ax.tick_params(colors=P["muted"])
    plt.tight_layout()

    fname = os.path.join(out_dir, f"cm_{name.lower().replace(' ', '_')}.png")
    fig.savefig(fname, dpi=150, bbox_inches="tight", facecolor=P["bg"])
    plt.close(fig)
    print(f"Saved: {fname}")

# ---------------------------------------------------------------------
# 2) FEATURE IMPORTANCE PLOT
#    - tree models: use feature_importances_
#    - non-tree models: fallback to permutation importance
# ---------------------------------------------------------------------
best_model_name = best_name
best_model = results[best_model_name]["model"]

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    importance_source = "Native feature importance"
else:
    print(f"{best_model_name} has no native feature_importances_. Using permutation importance...")
    perm = permutation_importance(
        best_model,
        X_te_scaled,
        y_te,
        n_repeats=30,
        random_state=SEED,
        scoring="roc_auc",
        n_jobs=-1
    )
    importances = perm.importances_mean
    importance_source = "Permutation importance"

imp_df = pd.DataFrame({
    "Feature": final_features,
    "Importance": importances
}).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(12, max(7, len(final_features) * 0.45 + 2)), facecolor=P["bg"])
ax.set_facecolor(P["card"])

bars = ax.barh(
    imp_df["Feature"],
    imp_df["Importance"],
    color=P["accent"],
    edgecolor=P["border"],
    height=0.65
)

for bar, val in zip(bars, imp_df["Importance"]):
    ax.text(
        bar.get_width() + 0.001,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}",
        va="center",
        fontsize=8,
        color=P["muted"]
    )

ax.set_xlabel("Importance")
ax.set_title(f"Feature Importance — {best_model_name} ({importance_source})", color=P["text"], fontweight="bold")
ax.grid(axis="x")
plt.tight_layout()

fname = os.path.join(out_dir, f"feature_importance_{best_model_name.lower().replace(' ', '_')}.png")
fig.savefig(fname, dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close(fig)
print(f"Saved: {fname}")

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "="*55)
print("IMPROVEMENTS APPLIED")
print("  1. KNN Imputer (k=5, distance-weighted) instead of median/mode")
print("  2. TARGET_N raised 200 → 300 (more augmented samples)")
print("  3. YJO fitness: Accuracy → AUC  (more discriminative)")
print("  4. YJO wasps/iters: 10/12 → 15/20 (Stage1), 8/10 → 12/15 (Stage2)")
print("  5. BorderlineSMOTE instead of standard SMOTE")
print("  6. RobustScaler on continuous features (critical for KNN)")
print("  7. Threshold: F1-only → Youden's J (sensitivity + specificity - 1)")
print("  8. RandomizedSearchCV n_iter: 60 → 100 + wider hyperparameter grids")
print("  9. RepeatedStratifiedKFold (5×3) for threshold optimisation")
print("="*55)
print("DONE")
print(f"  Real train / test split : {len(X_train_r)} / {len(X_test_r)}")
print(f"  Augmented training size : {len(X_aug)}")
print(f"  Final features          : {len(final_features)}")
print(f"\n  -- Test-set results (real held-out patients, Youden threshold) --")
for name, r in results.items():
    mark = "  [BEST AUC]" if name == best_name else ""
    print(f"  {name:<18}  Acc={r['accuracy']:.3f}  AUC={r['auc']:.3f}  "
          f"Rec={r['recall']:.3f}  F1={r['f1']:.3f}  thresh={r['threshold']:.2f}{mark}")
print()


# =============================================================================
# SAVE MODEL FOR WEB DEPLOYMENT
# =============================================================================

import joblib
import os

SAVE_DIR = r"D:\medical_1D\saved_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# -------------------------------------------------------
# Best model
# -------------------------------------------------------
best_model = results[best_name]["model"]

joblib.dump(best_model,
            os.path.join(SAVE_DIR, "popf_model.pkl"))

print("✔ Best model saved.")

# -------------------------------------------------------
# Scaler
# -------------------------------------------------------
joblib.dump(scaler,
            os.path.join(SAVE_DIR, "scaler.pkl"))

print("✔ Scaler saved.")

# -------------------------------------------------------
# Feature names
# -------------------------------------------------------
joblib.dump(final_features,
            os.path.join(SAVE_DIR, "features.pkl"))

print("✔ Feature list saved.")

# -------------------------------------------------------
# Prediction threshold
# -------------------------------------------------------
joblib.dump(results[best_name]["threshold"],
            os.path.join(SAVE_DIR, "threshold.pkl"))

print("✔ Threshold saved.")

# -------------------------------------------------------
# Clinical Knowledge Dictionary
# -------------------------------------------------------
joblib.dump(CLINICAL_KNOWLEDGE,
            os.path.join(SAVE_DIR, "clinical_knowledge.pkl"))

print("✔ Clinical knowledge saved.")

# -------------------------------------------------------
# Non-traditional Features
# -------------------------------------------------------
joblib.dump(NON_TRADITIONAL,
            os.path.join(SAVE_DIR, "non_traditional.pkl"))

print("✔ Non-traditional feature dictionary saved.")

# -------------------------------------------------------
# Binary Columns
# -------------------------------------------------------
joblib.dump(binary_cols,
            os.path.join(SAVE_DIR, "binary_columns.pkl"))

print("✔ Binary columns saved.")

# -------------------------------------------------------
# Continuous Columns
# -------------------------------------------------------
joblib.dump(continuous_cols,
            os.path.join(SAVE_DIR, "continuous_columns.pkl"))

print("✔ Continuous columns saved.")

# -------------------------------------------------------
# Original Feature Order
# -------------------------------------------------------
joblib.dump(feature_names,
            os.path.join(SAVE_DIR, "all_features.pkl"))

print("✔ Original feature order saved.")

# -------------------------------------------------------
# KNN Imputer
# -------------------------------------------------------
joblib.dump(knn_imp,
            os.path.join(SAVE_DIR, "knn_imputer.pkl"))

print("✔ KNN Imputer saved.")

# -------------------------------------------------------
# Deployment Information
# -------------------------------------------------------
deployment_info = {
    "best_model": best_name,
    "auc": results[best_name]["auc"],
    "accuracy": results[best_name]["accuracy"],
    "precision": results[best_name]["precision"],
    "recall": results[best_name]["recall"],
    "f1": results[best_name]["f1"],
    "threshold": results[best_name]["threshold"]
}

joblib.dump(deployment_info,
            os.path.join(SAVE_DIR, "deployment_info.pkl"))

print("✔ Deployment information saved.")

print("\n========================================")
print("ALL FILES SAVED SUCCESSFULLY")
print("Location:", SAVE_DIR)
print("========================================")

Raw: (76, 54)

── KNN Imputation (k=5) for missing values ──
Binary columns  (mode-imputed) : 7
Continuous cols (KNN-imputed)  : 20

Original: 74 patients | POPF: 43  No-POPF: 31

STEP 2 — 80/20 TRAIN/TEST SPLIT (before augmentation)
Train (real): 59 patients | POPF=34  No-POPF=25
Test  (real): 15  patients | POPF=9  No-POPF=6
  Test set = real patients only, never used during augmentation or tuning

── Linear interpolation augmentation (train only) ──
Real train : 59  |  Augmented train : 300  |  Test (real) : 15
  Binary column integrity verified

STAGE 1 — YJO on all features  [n_wasps=15, n_iter=20, AUC fitness]
  [Stage-1] Init  best=0.9998  features=13
  [Stage-1] Iter  1/20  p_attack=0.20  best=1.0000  features=13
  [Stage-1] Iter  5/20  p_attack=0.32  best=1.0000  features=13
  [Stage-1] Iter 10/20  p_attack=0.46  best=1.0000  features=13
  [Stage-1] Iter 15/20  p_attack=0.61  best=1.0000  features=13
  [Stage-1] Iter 20/20  p_attack=0.75  best=1.0000  features=13
  [Stage-1] D

In [1]:
import warnings; warnings.filterwarnings("ignore")
import re, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import shap

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import KNNImputer
from sklearn.model_selection import (StratifiedKFold,
                                     train_test_split, RandomizedSearchCV,
                                     cross_val_predict)
from sklearn.metrics import (classification_report, accuracy_score,
                              roc_auc_score, confusion_matrix,
                              precision_score, recall_score, f1_score)
from imblearn.over_sampling import BorderlineSMOTE
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("⚠ XGBoost not installed — skipping XGBoost model")

SEED = 42
np.random.seed(SEED)

# ── Palette ──────────────────────────────────────────────────────────────────
P = {"bg":"#0F1117","card":"#1A1D27","accent":"#7C6AF7","green":"#4ADE80",
     "red":"#F87171","text":"#E8E8F0","muted":"#8B8FA8","border":"#2A2D3E",
     "orange":"#FB923C","blue":"#60A5FA","yellow":"#FCD34D"}

plt.rcParams.update({
    "figure.facecolor":P["bg"],"axes.facecolor":P["card"],
    "axes.edgecolor":P["border"],"axes.labelcolor":P["text"],
    "xtick.color":P["muted"],"ytick.color":P["muted"],"text.color":P["text"],
    "grid.color":P["border"],"grid.linestyle":"--","grid.alpha":0.4,
    "font.family":"sans-serif","axes.titlesize":11,"axes.labelsize":9,
})

# ── Non-traditional features (paper novelty) ─────────────────────────────────
NON_TRADITIONAL = {
    "hb"             : "Haemoglobin (g/dL)",
    "albumin "       : "Serum Albumin (g/dL)",
    "ALP"            : "Alkaline Phosphatase (U/L)",
    "HBA1C"          : "HbA1c (%)",
    "bill "          : "Serum Bilirubin (mg/dL)",
    "BLOOD LOSS "    : "Intra-op Blood Loss (mL)",
    "CBD DIAMETER"   : "CBD Diameter (mm)",
    "ca 19-9"        : "CA 19-9 (U/mL)",
    "Cea"            : "CEA (ng/mL)",
    "fever "         : "Pre-op Fever",
    "RECENT DIABETES": "Recent Diabetes",
    "comorbidity"    : "Comorbidity",
    "neoadjuvant "   : "Neoadjuvant Treatment",
    "ercp stenting " : "ERCP Stenting",
    "jaundice"       : "Obstructive Jaundice",
    "pain "          : "Pre-op Pain",
    "weight loss "   : "Weight Loss",
    "bleeding "      : "Pre-op Bleeding",
    "Age "           : "Patient Age",
    "Sex"            : "Patient Sex",
}

# =============================================================================
# STEP 1 — LOAD & CLEAN
# =============================================================================
df = pd.read_excel("D:\\medical_1D\\1D data 2.xlsx")
print(f"Raw: {df.shape}")

DROP = ["DGE","PPH","SSI","CLAVIEN DINDO","BILE LAK","HOSPITAL STAY ",
        "READMISION","RESURGERY","mortality",
        "s no ","Name ","Address","CR Number ",
        "Unnamed: 5","Unnamed: 6","Unnamed: 7",
        "date of admisson ","date of sx ","date of discharge ","date of surgery",
        "cect","eus ","periampullary carcinoma ","surgery","CHROMAGANIN "]
df.drop(columns=[c for c in DROP if c in df.columns], inplace=True)

df.dropna(subset=["POPF"], inplace=True)
df["POPF"] = (df["POPF"] != 0).astype(int)

def auto_encode(col, series):
    name = col.strip().lower()
    if series.dtype in [np.float64, np.int64, float, int]:
        num = pd.to_numeric(series, errors="coerce")
        return num if num.notna().mean() >= 0.4 else None
    s = series.astype(str).str.strip()
    if name == "lap open": return None
    if name in ("fever ", "neoadjuvant "):
        def _b(v):
            sv = str(v).strip()
            if sv in ("-","9","nan",""): return np.nan
            try:   return 1.0 if float(sv) > 0 else 0.0
            except: return np.nan
        return series.apply(_b)
    if name == "ercp stenting ":
        return series.apply(lambda v: 1.0 if str(v).strip().startswith("1")
                            else (0.0 if str(v).strip() == "0" else np.nan))
    if name == "comorbidity":
        return series.apply(lambda v: 0.0 if str(v).strip() in ("0","nan","")
                            else (0.0 if str(v).strip() == "0" else 1.0))
    if name == "bill ":
        return pd.to_numeric(s.str.replace(r"[Oo]","0",regex=True), errors="coerce")
    if name == "cea":
        return pd.to_numeric(s.str.replace("`",""), errors="coerce")
    if name == "tumor size":
        return series.apply(lambda v: max(
            (float(n) for n in re.findall(r"[\d.]+", str(v))), default=np.nan))
    num = pd.to_numeric(s.str.replace("`","").str.replace(",","."), errors="coerce")
    return num if num.notna().mean() >= 0.4 else None

y   = df["POPF"].astype(int)
raw = df.drop(columns=["POPF"])
keep = {}
for col in raw.columns:
    res = auto_encode(col, raw[col])
    if res is not None and res.notna().mean() >= 0.20 and res.nunique() > 1:
        keep[col] = res.astype(float)

X_df = pd.DataFrame(keep)
feature_names = list(X_df.columns)

def is_binary_col(series):
    vals = set(series.dropna().unique())
    return vals.issubset({0.0, 1.0})

binary_cols     = [c for c in X_df.columns if is_binary_col(X_df[c])]
continuous_cols = [c for c in X_df.columns if c not in binary_cols]

# ── IMPROVEMENT 1: KNN Imputer for smarter missing value filling ──────────────
print("\n── KNN Imputation (k=5) for missing values ──")
knn_imp = KNNImputer(n_neighbors=5, weights="distance")

# Impute continuous cols with KNN, binary with mode (preserves 0/1 integrity)
X_cont = X_df[continuous_cols].values
X_cont_imputed = knn_imp.fit_transform(X_cont)
for i, col in enumerate(continuous_cols):
    X_df[col] = X_cont_imputed[:, i]
for col in binary_cols:
    X_df[col].fillna(X_df[col].mode()[0], inplace=True)

print(f"Binary columns  (mode-imputed) : {len(binary_cols)}")
print(f"Continuous cols (KNN-imputed)  : {len(continuous_cols)}")

X_np = X_df.values.copy()
y_np = y.values.copy()
binary_idx     = [feature_names.index(c) for c in binary_cols]
continuous_idx = [feature_names.index(c) for c in continuous_cols]
print(f"\nOriginal: {len(y_np)} patients | POPF: {y_np.sum()}  No-POPF: {(y_np==0).sum()}")

# =============================================================================
# STEP 2 — 80/20 TRAIN/TEST SPLIT (BEFORE AUGMENTATION — no leakage)
# =============================================================================
print("\n" + "="*55)
print("STEP 2 — 80/20 TRAIN/TEST SPLIT (before augmentation)")
print("="*55)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_np, y_np, test_size=0.20, stratify=y_np, random_state=SEED)
print(f"Train (real): {len(y_train_r)} patients | POPF={y_train_r.sum()}  No-POPF={(y_train_r==0).sum()}")
print(f"Test  (real): {len(y_test_r)}  patients | POPF={y_test_r.sum()}  No-POPF={(y_test_r==0).sum()}")
print("  Test set = real patients only, never used during augmentation or tuning")

# =============================================================================
# STEP 3 — BINARY-SAFE INTERPOLATION AUGMENTATION (training set only)
# ── IMPROVEMENT 2: TARGET_N raised 200→300 for more robust training ──────────
# =============================================================================
print("\n── Linear interpolation augmentation (train only) ──")

TARGET_N = 300                      # ← raised from 200
cont_idx = [i for i in range(X_np.shape[1]) if i not in binary_idx]
rng_aug  = np.random.default_rng(SEED)

def linear_interpolate_class(X_class, n_needed, binary_idx, cont_idx, rng):
    n_real, n_feat = X_class.shape
    X_syn = np.empty((n_needed, n_feat), dtype=float)
    for s in range(n_needed):
        a, b = rng.choice(n_real, size=2, replace=(n_real < 2))
        lam  = rng.uniform(0.0, 1.0)
        X_syn[s, cont_idx]   = lam * X_class[a, cont_idx] + (1.0-lam) * X_class[b, cont_idx]
        X_syn[s, binary_idx] = np.where(lam >= 0.5,
                                         X_class[a, binary_idx],
                                         X_class[b, binary_idx])
    return X_syn

idx_pos = np.where(y_train_r == 1)[0]
idx_neg = np.where(y_train_r == 0)[0]
X_pos, X_neg = X_train_r[idx_pos], X_train_r[idx_neg]
half      = TARGET_N // 2
n_syn_pos = max(0, half - len(X_pos))
n_syn_neg = max(0, half - len(X_neg))
X_syn_pos = linear_interpolate_class(X_pos, n_syn_pos, binary_idx, cont_idx, rng_aug)
X_syn_neg = linear_interpolate_class(X_neg, n_syn_neg, binary_idx, cont_idx, rng_aug)

X_aug = np.vstack([X_train_r, X_syn_pos, X_syn_neg])
y_aug = np.concatenate([y_train_r,
                        np.ones(n_syn_pos, dtype=int),
                        np.zeros(n_syn_neg, dtype=int)])

print(f"Real train : {len(X_train_r)}  |  Augmented train : {len(X_aug)}  |  Test (real) : {len(X_test_r)}")

for idx in binary_idx:
    vals = set(np.unique(X_aug[:, idx]))
    assert vals.issubset({0.0, 1.0}), f"Binary violated: {feature_names[idx]}"
print("  Binary column integrity verified")

# =============================================================================
def crow_search_optimizer(X_data, y_data, n_crows, n_iter,
                         awareness_prob=0.2, flight_length=2,
                         min_feat=3, label="CSA"):
    
    n_feat = X_data.shape[1]
    rng = np.random.default_rng(SEED)

    # ── Fitness (same as your YJO: AUC-based) ─────────────────────────
    def fitness(mask):
        if mask.sum() == 0:
            return 0.0

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        aucs = []

        for tr, te in skf.split(X_data[:, mask], y_data):
            Xtr, Xte = X_data[tr][:, mask], X_data[te][:, mask]
            ytr, yte = y_data[tr], y_data[te]

            k = min(4, int(ytr.sum()) - 1)
            if k < 1:
                continue

            try:
                Xs, ys = BorderlineSMOTE(random_state=SEED, k_neighbors=k).fit_resample(Xtr, ytr)
            except:
                Xs, ys = Xtr, ytr

            model = RandomForestClassifier(
                n_estimators=50,
                max_depth=5,
                class_weight="balanced",
                random_state=SEED
            )
            model.fit(Xs, ys)

            prob = model.predict_proba(Xte)[:, 1]
            if len(np.unique(yte)) > 1:
                aucs.append(roc_auc_score(yte, prob))

        return float(np.mean(aucs)) if aucs else 0.0

    # ── Ensure minimum features ───────────────────────────────────────
    def enforce_min(mask):
        if mask.sum() < min_feat:
            off = np.where(mask == 0)[0]
            need = min_feat - int(mask.sum())
            mask[rng.choice(off, min(need, len(off)), replace=False)] = 1
        return mask

    # ── Initialize population ─────────────────────────────────────────
    positions = rng.integers(0, 2, (n_crows, n_feat)).astype(float)
    positions = np.array([enforce_min(p) for p in positions])

    memory = positions.copy()
    mem_scores = np.array([fitness(p.astype(bool)) for p in positions])

    best_idx = np.argmax(mem_scores)
    g_best = memory[best_idx].copy()
    g_score = mem_scores[best_idx]

    history = [g_score]
    print(f"  [{label}] Init best={g_score:.4f} features={int(g_best.sum())}")

    # ── CSA Main Loop ─────────────────────────────────────────────────
    for it in range(n_iter):
        for i in range(n_crows):

            j = rng.integers(0, n_crows)  # random crow to follow

            if rng.random() > awareness_prob:
                # Follow another crow's memory
                step = rng.uniform(0, 1, n_feat)
                new_pos = positions[i] + step * flight_length * (memory[j] - positions[i])
            else:
                # Random movement (exploration)
                new_pos = rng.integers(0, 2, n_feat)

            # Convert to binary
            new_pos = (new_pos > 0.5).astype(float)
            new_pos = enforce_min(new_pos)

            score = fitness(new_pos.astype(bool))

            # Update memory
            if score > mem_scores[i]:
                memory[i] = new_pos.copy()
                mem_scores[i] = score

            # Update global best
            if score > g_score:
                g_best = new_pos.copy()
                g_score = score

            positions[i] = new_pos

        history.append(g_score)

        if (it + 1) % 5 == 0 or it == 0:
            print(f"  [{label}] Iter {it+1}/{n_iter}  best={g_score:.4f} features={int(g_best.sum())}")

    print(f"  [{label}] Done best={g_score:.4f} features={int(g_best.sum())}")
    return g_best.astype(bool), g_score, history

print("\n" + "="*55)
print("STAGE 1 — csa on all features  [n_wasps=15, n_iter=20, AUC fitness]")
print("="*55)
mask_A, score_A, hist_A = crow_search_optimizer(
    X_aug, y_aug,
    n_crows=15,
    n_iter=20,
    awareness_prob=0.2,
    flight_length=2,
    min_feat=4,
    label="CSA Stage-1"
)
set_A = [feature_names[i] for i, m in enumerate(mask_A) if m]
set_B = [feature_names[i] for i, m in enumerate(mask_A) if not m]
print(f"\n  Set A (important)  : {len(set_A)}  ->  {set_A}")
print(f"  Set B (unimportant): {len(set_B)}  ->  {set_B}")

print("\n" + "="*55)
print("STAGE 2 — csa on unimportant Set B  [n_wasps=12, n_iter=15]")
print("="*55)
idx_B = [i for i, m in enumerate(mask_A) if not m]

mask_C, score_C, hist_C = crow_search_optimizer(
    X_aug[:, idx_B], y_aug,
    n_crows=12,
    n_iter=15,
    awareness_prob=0.25,
    flight_length=3,
    min_feat=2,
    label="CSA Stage-2"
)
set_C = [set_B[i] for i, m in enumerate(mask_C) if m]
print(f"\n  Set C (rescued): {len(set_C)}  ->  {set_C}")

final_features = list(dict.fromkeys(set_A + set_C))
print(f"\nFinal features (A + C): {len(final_features)}")
for f in final_features:
    tag = "  [NON-TRAD]" if f in NON_TRADITIONAL else ""
    src = "A" if f in set_A else "C"
    print(f"  [{src}]  {f}{tag}")

# =============================================================================
# STEP 5 — MULTI-MODEL TRAINING + DEEP HYPERPARAMETER TUNING
# ── IMPROVEMENT 4: RobustScaler pipelines, BorderlineSMOTE, Youden threshold,
#                   n_iter=100, wider grids, RepeatedStratifiedKFold ──────────
# =============================================================================
feat_idx = [feature_names.index(f) for f in final_features]
X_tr     = X_aug[:, feat_idx].copy()
y_tr     = y_aug.copy()
X_te     = X_test_r[:, feat_idx].copy()
y_te     = y_test_r.copy()

# ── Feature scaling on continuous columns only ────────────────────────────────
final_cont_mask = np.array([f not in binary_cols for f in final_features])
scaler = RobustScaler()

def scale_data(X_train, X_test, cont_mask):
    """Fit scaler on train continuous cols, transform both — no leakage."""
    X_tr_s = X_train.copy()
    X_te_s = X_test.copy()
    if cont_mask.any():
        X_tr_s[:, cont_mask] = scaler.fit_transform(X_train[:, cont_mask])
        X_te_s[:, cont_mask] = scaler.transform(X_test[:, cont_mask])
    return X_tr_s, X_te_s

X_tr_scaled, X_te_scaled = scale_data(X_tr, X_te, final_cont_mask)

# ── BorderlineSMOTE on augmented training set ─────────────────────────────────
k_sm = min(4, int(y_tr.sum()) - 1)
try:
    Xs_tr, ys_tr = BorderlineSMOTE(random_state=SEED, k_neighbors=k_sm).fit_resample(
        X_tr_scaled, y_tr)
except Exception:
    from imblearn.over_sampling import SMOTE
    Xs_tr, ys_tr = SMOTE(random_state=SEED, k_neighbors=k_sm).fit_resample(
        X_tr_scaled, y_tr)

neg_count = int((ys_tr == 0).sum())
pos_count = int((ys_tr == 1).sum())
spw       = round(neg_count / max(pos_count, 1), 2)
print(f"\nBorderlineSMOTE output -> neg={neg_count}  pos={pos_count}  scale_pos_weight={spw}")

print("\n" + "="*55)
print("STEP 5 — MULTI-MODEL TRAINING + DEEP HYPERPARAMETER TUNING")
print(f"  Train : {len(Xs_tr)} (BorderlineSMOTE on augmented) | Test : {len(X_te_scaled)} (real)")
print(f"  Scaling: RobustScaler on {final_cont_mask.sum()} continuous features")
print("="*55)

# ── IMPROVEMENT 5: Youden's J threshold (sensitivity + specificity − 1) ──────
def best_threshold_youden(model, X, y):
    """
    Find threshold maximising Youden's J = Sensitivity + Specificity − 1.
    More robust than F1-only for imbalanced medical data.
    Note: cross_val_predict requires partitions so we use StratifiedKFold(10).
    """
    skf      = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
    cv_prob  = cross_val_predict(model, X, y, cv=skf, method="predict_proba")[:, 1]
    thresholds = np.arange(0.20, 0.81, 0.02)
    best_t, best_j = 0.5, -1.0
    for t in thresholds:
        yp  = (cv_prob >= t).astype(int)
        rec = recall_score(y, yp, zero_division=0)                    # sensitivity
        tn  = confusion_matrix(y, yp).ravel()
        # specificity = TN / (TN + FP)
        cm  = confusion_matrix(y, yp)
        if cm.shape == (2, 2):
            spec = cm[0, 0] / max(cm[0, 0] + cm[0, 1], 1)
        else:
            spec = 0.0
        j = rec + spec - 1.0
        if j > best_j:
            best_j = j
            best_t = t
    return float(best_t)

# ── Model definitions ─────────────────────────────────────────────────────────
MODELS = {}

# 1. Random Forest — wider grid, more estimators
MODELS["Random Forest"] = (
    RandomForestClassifier(random_state=SEED),
    {
        "n_estimators": [500, 700, 1000],
        "max_depth": [6, 8, 10, 12, None],
        "min_samples_leaf"  : [1, 2],
        "min_samples_split" : [2,3],
        "max_features"      : ["sqrt",0.7,None],
        "class_weight"      : ["balanced", "balanced_subsample",
                               {0:1,1:2}, {0:1,1:3}, {0:1,1:4}],
        "max_samples"       : [0.7, 0.8, 0.9, None],
    }
)

# 2. Decision Tree
MODELS["Decision Tree"] = (
    DecisionTreeClassifier(random_state=SEED),
    {
        "max_depth"         : [3, 4, 5, 6, 7, 8, None],
        "min_samples_leaf"  : [1, 2, 3, 5],
        "min_samples_split" : [2, 4, 6],
        "criterion"         : ["gini", "entropy"],
        "class_weight"      : ["balanced", {0:1,1:2}, {0:1,1:3}, {0:1,1:4}],
        "ccp_alpha"         : [0.0, 0.002, 0.005, 0.01, 0.02, 0.05],
        "splitter"          : ["best", "random"],
    }
)

# 3. XGBoost — deeper regularisation grid
if HAS_XGB:
    MODELS["XGBoost"] = (
        XGBClassifier(random_state=SEED, eval_metric="logloss", verbosity=0),
        {
            "n_estimators"     : [100, 200, 300, 500, 700],
            "max_depth"        : [3, 4, 5, 6, 7],
            "learning_rate"    : [0.005, 0.01, 0.05, 0.1, 0.2],
            "subsample"        : [0.6, 0.7, 0.8, 0.9, 1.0],
            "colsample_bytree" : [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            "min_child_weight" : [1, 2, 3, 5, 7],
            "gamma"            : [0, 0.05, 0.1, 0.2, 0.3, 0.5],
            "reg_alpha"        : [0, 0.01, 0.05, 0.1, 0.5, 1.0],
            "reg_lambda"       : [0.5, 1.0, 1.5, 2.0, 3.0],
            "scale_pos_weight" : [1.0, 1.5, spw, spw*1.5, spw*2.0],
            "max_delta_step"   : [0, 1, 5],   # helps with imbalanced data
        }
    )

# 4. KNN — uses scaled data, much wider metric/neighbour grid
MODELS["KNN"] = (
    KNeighborsClassifier(),
    {
        "n_neighbors": [15, 21,25],
        "weights"    : ["distance"],
        "metric"     : ["euclidean", "manhattan"],
        "p"          : [1, 2, 3],
        "leaf_size"  : [10, 20, 30, 40, 50],
        "algorithm"  : ["auto", "ball_tree", "kd_tree"],
    }
)

# ── Train + tune + threshold-optimise each model ──────────────────────────────
results = {}
best_rf_model = None

print(f"\n{'Model':<18} {'Best AUC(CV)':>12} {'Thresh':>7} "
      f"{'Test Acc':>9} {'Test AUC':>9} {'Prec':>7} {'Rec':>7} {'F1':>7}")
print("─" * 82)

# ── IMPROVEMENT 6: n_iter=100 (from 60) for more thorough search ──────────────
SEARCH_ITER = 100

for name, (base_model, param_grid) in MODELS.items():
    search = RandomizedSearchCV(
        base_model, param_grid,
        n_iter=SEARCH_ITER, cv=5, scoring="roc_auc",
        n_jobs=-1, refit=True, random_state=SEED
    )
    search.fit(Xs_tr, ys_tr)
    best_model = search.best_estimator_

    # Youden's J threshold on training CV
    thresh = best_threshold_youden(best_model, Xs_tr, ys_tr)

    # Evaluate on real held-out test set (scaled)
    ypr = best_model.predict_proba(X_te_scaled)[:, 1]
    yp  = (ypr >= thresh).astype(int)

    acc  = accuracy_score(y_te, yp)
    auc  = roc_auc_score(y_te, ypr) if len(np.unique(y_te)) > 1 else np.nan
    prec = precision_score(y_te, yp, zero_division=0)
    rec  = recall_score(y_te, yp, zero_division=0)
    f1   = f1_score(y_te, yp, zero_division=0)

    results[name] = {
        "model"       : best_model,
        "best_params" : search.best_params_,
        "cv_auc"      : search.best_score_,
        "threshold"   : thresh,
        "y_pred"      : yp,
        "y_prob"      : ypr,
        "accuracy"    : acc,
        "auc"         : auc,
        "precision"   : prec,
        "recall"      : rec,
        "f1"          : f1,
    }

    print(f"{name:<18} {search.best_score_:>12.3f} {thresh:>7.2f} "
          f"{acc:>9.3f} {auc:>9.3f} {prec:>7.3f} {rec:>7.3f} {f1:>7.3f}")

    if name == "Random Forest":
        best_rf_model = best_model

# ── Detailed reports ──────────────────────────────────────────────────────────
print("\n" + "="*55)
print("DETAILED CLASSIFICATION REPORTS (real held-out test set)")
print("="*55)
for name, r in results.items():
    print(f"\n-- {name} --")
    print(f"  Best params : {r['best_params']}")
    print(f"  Threshold   : {r['threshold']:.2f}  (Youden's J on training CV)")
    print(classification_report(y_te, r["y_pred"], target_names=["No POPF","POPF"]))

best_name = max(results, key=lambda n: results[n]["auc"])
print(f"\n  Best model by Test AUC: {best_name}  (AUC={results[best_name]['auc']:.3f})")

# =============================================================================
# STEP 6 — SHAP + CLINICAL EXPLAINER
# (SHAP fitted on UNscaled X_tr so feature values remain interpretable)
# =============================================================================
CLINICAL_KNOWLEDGE = {
    "hb": {
        "display"    : "Haemoglobin",
        "unit"       : "g/dL",
        "low_thresh" : 10.0, "high_thresh": None,
        "low_status" : "Anaemic", "high_status": "Normal",
        "normal_range": "12-16 g/dL (F), 13-17 g/dL (M)",
        "low_risk_dir": "increases POPF risk", "high_risk_dir": "decreases POPF risk",
        "why_it_matters": (
            "Low haemoglobin indicates pre-operative anaemia and tissue oxygen deprivation. "
            "Poorly oxygenated pancreatic tissue heals slowly, weakening the anastomosis "
            "and making a pancreatic fistula more likely."),
        "doctor_note": "Doctors routinely check Hb but rarely link it to POPF risk.",
    },
    "albumin ": {
        "display"    : "Serum Albumin",
        "unit"       : "g/dL",
        "low_thresh" : 3.5, "high_thresh": None,
        "low_status" : "Hypoalbuminaemia", "high_status": "Normal",
        "normal_range": "3.5-5.0 g/dL",
        "low_risk_dir": "increases POPF risk", "high_risk_dir": "decreases POPF risk",
        "why_it_matters": (
            "Albumin is the primary marker of nutritional status. Hypoalbuminaemia "
            "impairs wound healing, reduces anastomotic tensile strength, and increases "
            "susceptibility to post-operative leaks."),
        "doctor_note": "Serum albumin is checked pre-op but not integrated into POPF risk scoring.",
    },
    "ALP": {
        "display"    : "Alkaline Phosphatase (ALP)",
        "unit"       : "U/L",
        "low_thresh" : None, "high_thresh": 120.0,
        "low_status" : "Normal", "high_status": "Elevated",
        "normal_range": "44-147 U/L",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Elevated ALP reflects biliary obstruction and hepatic stress. Biliary "
            "hypertension raises intra-ductal pancreatic pressure, compromising "
            "anastomotic integrity and increasing fistula risk."),
        "doctor_note": "ALP is ordered as part of a liver panel but not used in POPF prediction.",
    },
    "HBA1C": {
        "display"    : "HbA1c (Glycated Haemoglobin)",
        "unit"       : "%",
        "low_thresh" : None, "high_thresh": 6.5,
        "low_status" : "Normal glycaemic control", "high_status": "Poor glycaemic control",
        "normal_range": "< 5.7% normal, 5.7-6.4% pre-diabetic, >= 6.5% diabetic",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Chronically elevated blood sugar causes microvascular damage and impairs "
            "neutrophil function, delaying anastomotic healing and increasing "
            "vulnerability to post-operative complications."),
        "doctor_note": "HbA1c is checked for diabetics but overlooked as a surgical risk factor.",
    },
    "bill ": {
        "display"    : "Serum Bilirubin",
        "unit"       : "mg/dL",
        "low_thresh" : None, "high_thresh": 2.0,
        "low_status" : "Normal", "high_status": "Hyperbilirubinaemia",
        "normal_range": "0.2-1.2 mg/dL",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Pre-operative jaundice and elevated bilirubin cause Kupffer-cell "
            "dysfunction and endotoxaemia, impairing the immune response and "
            "increasing post-operative fistula complications."),
        "doctor_note": "Bilirubin is well-monitored for biliary disease but not in POPF models.",
    },
    "BLOOD LOSS ": {
        "display"    : "Intra-operative Blood Loss",
        "unit"       : "mL",
        "low_thresh" : None, "high_thresh": 500.0,
        "low_status" : "Acceptable", "high_status": "Excessive",
        "normal_range": "< 500 mL acceptable for Whipple",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Excessive intra-operative haemorrhage triggers coagulopathy and "
            "systemic inflammatory response, impairing anastomotic integrity "
            "and increasing the likelihood of fistula formation."),
        "doctor_note": "Blood loss is recorded but not used quantitatively in POPF prediction.",
    },
    "CBD DIAMETER": {
        "display"    : "Common Bile Duct Diameter",
        "unit"       : "mm",
        "low_thresh" : None, "high_thresh": 8.0,
        "low_status" : "Normal", "high_status": "Dilated",
        "normal_range": "< 6-8 mm",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "A dilated CBD indicates chronic biliary obstruction, leading to hepatic "
            "reserve impairment and elevated intra-ductal pressure, compromising "
            "anastomotic healing after pancreaticoduodenectomy."),
        "doctor_note": "CBD diameter is measured on imaging but rarely used as a POPF predictor.",
    },
    "ca 19-9": {
        "display"    : "CA 19-9 Tumour Marker",
        "unit"       : "U/mL",
        "low_thresh" : None, "high_thresh": 37.0,
        "low_status" : "Normal", "high_status": "Elevated",
        "normal_range": "< 37 U/mL",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Elevated CA 19-9 reflects tumour burden and associated inflammatory "
            "milieu, which may compromise tissue quality and anastomotic healing."),
        "doctor_note": "CA 19-9 is used for diagnosis but not for POPF prediction.",
    },
    "weight loss ": {
        "display"    : "Pre-operative Weight Loss",
        "unit"       : "(binary: 1=yes)",
        "low_thresh" : None, "high_thresh": None,
        "low_status" : "No weight loss", "high_status": "Significant weight loss",
        "normal_range": "Absent",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Pre-operative weight loss is a surrogate for cachexia and protein-energy "
            "malnutrition. Malnourished patients have impaired wound healing and higher "
            "anastomotic complication rates."),
        "doctor_note": "Weight loss is documented clinically but not formally scored for POPF.",
    },
    "fever ": {
        "display"    : "Pre-operative Fever",
        "unit"       : "(binary: 1=yes)",
        "low_thresh" : None, "high_thresh": None,
        "low_status" : "Afebrile", "high_status": "Febrile",
        "normal_range": "Absent",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Pre-operative fever indicates active infection or systemic inflammation. "
            "Operating in an inflammatory state predisposes to impaired anastomotic "
            "healing and higher fistula rates."),
        "doctor_note": "Fever is a general surgical risk factor but not in standard POPF scores.",
    },
}

def get_status(feat, value):
    info = CLINICAL_KNOWLEDGE.get(feat)
    if not info: return "--"
    if info["low_thresh"] and value < info["low_thresh"]:
        return f"{info['low_status']} (< {info['low_thresh']} {info['unit']})"
    if info["high_thresh"] and value > info["high_thresh"]:
        return f"{info['high_status']} (> {info['high_thresh']} {info['unit']})"
    return f"Normal range ({info['normal_range']})"

# Fit SHAP on unscaled training data (interpretable feature values)
print("\n-- Computing SHAP (Random Forest, unscaled X_tr) --")
shap_rf    = best_rf_model
# Re-fit on unscaled data for SHAP interpretability
shap_rf.fit(
    *BorderlineSMOTE(random_state=SEED, k_neighbors=min(4,int(y_tr.sum())-1)).fit_resample(
        X_tr, y_tr)
    if min(4,int(y_tr.sum())-1) >= 1 else (X_tr, y_tr)
)
X_df_shap  = pd.DataFrame(X_tr, columns=final_features)
explainer  = shap.TreeExplainer(shap_rf)
sv         = explainer(X_df_shap)
sv_popf    = sv.values[:, :, 1]


def clinical_explain(idx, X_data, y_data, sv_data, model):
    sv_  = sv_data[idx]
    fv   = X_data[idx]
    prob = model.predict_proba(X_data[idx:idx+1])[0, 1]
    pred = "POPF" if prob >= 0.5 else "No POPF"
    true = "POPF" if y_data[idx] == 1 else "No POPF"

    nt_rows, tr_rows = [], []
    for fi, fname in enumerate(final_features):
        if fname in CLINICAL_KNOWLEDGE or fname in NON_TRADITIONAL:
            nt_rows.append((fi, fname, sv_[fi], fv[fi]))
        else:
            tr_rows.append((fi, fname, sv_[fi], fv[fi]))

    nt_rows.sort(key=lambda x: abs(x[2]), reverse=True)
    tr_rows.sort(key=lambda x: abs(x[2]), reverse=True)

    sep = "=" * 68
    out = (f"\n{sep}\n"
           f"  PATIENT {idx+1}  |  Prediction: {pred}  |  Actual: {true}\n"
           f"  POPF Risk Probability: {prob*100:.1f}%\n"
           f"{sep}\n\n")

    out += "  NON-TRADITIONAL FEATURES\n"
    out += "  " + "-" * 64 + "\n"
    for fi, fname, shap_val, val in nt_rows:
        info    = CLINICAL_KNOWLEDGE.get(fname, {})
        display = info.get("display", NON_TRADITIONAL.get(fname, fname))
        unit    = info.get("unit", "")
        status  = get_status(fname, val)
        arrow   = "increases POPF risk" if shap_val > 0 else "decreases POPF risk"
        out += f"\n  * {display}\n"
        out += f"    Value  : {val:.2f} {unit}\n"
        out += f"    Status : {status}\n"
        out += f"    Effect : {arrow}  (SHAP = {shap_val:+.4f})\n"
        if fname in CLINICAL_KNOWLEDGE:
            why = info["why_it_matters"]
            words = why.split(); line = "      "
            for w in words:
                if len(line) + len(w) > 66: out += line + "\n"; line = "      "
                line += w + " "
            out += line.rstrip() + "\n"

    out += f"\n  TRADITIONAL FEATURES (top 5)\n"
    out += "  " + "-" * 64 + "\n"
    for fi, fname, shap_val, val in tr_rows[:5]:
        arrow = "higher POPF" if shap_val > 0 else "lower POPF"
        out  += f"  . {fname.strip():<25} = {val:>8.2f}   SHAP {shap_val:+.4f}  {arrow}\n"
    return out

popf_idx   = np.where(y_tr == 1)[0][0]
nopopf_idx = np.where(y_tr == 0)[0][0]
print(clinical_explain(popf_idx,   X_tr, y_tr, sv_popf, shap_rf))
print(clinical_explain(nopopf_idx, X_tr, y_tr, sv_popf, shap_rf))

# =============================================================================
# PLOTS (3 figures)
# =============================================================================

# ── Plot 1: YJO Convergence ───────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(10, 4), facecolor=P["bg"])
ax2.set_facecolor(P["card"])
ax2.plot(hist_A, color=P["accent"], lw=2.5, marker="o", ms=5,
         label=f"Stage 1 — all {len(feature_names)} features")
ax2.plot(hist_C, color=P["orange"], lw=2.5, marker="s", ms=5, ls="--",
         label=f"Stage 2 — Set B ({len(set_B)} unimportant features)")
ax2.set_xlabel("Iteration"); ax2.set_ylabel("Best CV AUC")
ax2.set_title("csa Convergence — both stages (AUC fitness)", color=P["text"], fontweight="bold")
ax2.set_ylim(0.4, 1.05)
ax2.legend(facecolor=P["card"], edgecolor=P["border"], fontsize=9)
ax2.grid(True)
plt.tight_layout()
fig2.savefig("D:\\medical_1D\\new_output\\csa_convergence.png",
             dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close(fig2)
print("\nSaved: csa_convergence.png")

# ── Plot 2: Feature Importance ────────────────────────────────────────────────
imp_df = pd.DataFrame({
    "Feature"   : final_features,
    "Importance": shap_rf.feature_importances_,
    "Origin"    : ["A" if f in set_A else "C" for f in final_features],
    "Novel"     : [f in NON_TRADITIONAL for f in final_features],
}).sort_values("Importance", ascending=True)

fig3, ax3 = plt.subplots(figsize=(12, max(7, len(final_features)*0.45 + 2)),
                          facecolor=P["bg"])
ax3.set_facecolor(P["card"])
clrs = []
for _, row in imp_df.iterrows():
    if   row["Novel"] and row["Origin"] == "A": clrs.append(P["green"])
    elif row["Novel"] and row["Origin"] == "C": clrs.append(P["orange"])
    elif row["Origin"] == "A":                  clrs.append(P["blue"])
    else:                                        clrs.append(P["muted"])

bars = ax3.barh(imp_df["Feature"], imp_df["Importance"],
                color=clrs, edgecolor=P["border"], height=0.65, linewidth=0.4)
for bar, (_, row) in zip(bars, imp_df.iterrows()):
    lbl = f'{row["Importance"]:.3f}{"  [NT]" if row["Novel"] else ""}'
    ax3.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
             lbl, va="center", fontsize=8, color=P["muted"])
ax3.set_xlabel("Mean RF Importance (Gini)")
ax3.set_title("CSA-selected Features — importance x origin x novelty",
              color=P["text"], fontweight="bold")
ax3.grid(axis="x")
ax3.legend(handles=[
    mpatches.Patch(color=P["green"],  label="Non-traditional + Set A"),
    mpatches.Patch(color=P["orange"], label="Non-traditional + Set C (rescued)"),
    mpatches.Patch(color=P["blue"],   label="Traditional + Set A"),
    mpatches.Patch(color=P["muted"],  label="Traditional + Set C"),
], facecolor=P["card"], edgecolor=P["border"], fontsize=8, loc="lower right")
plt.tight_layout()
fig3.savefig("D:\\medical_1D\\new_output\\feature_importance_csa.png",
             dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close(fig3)
print("Saved: feature_importance_csa.png")

# ── Plot 3: Multi-Model Comparison ────────────────────────────────────────────
model_names = list(results.keys())
metrics     = ["accuracy", "auc", "precision", "recall", "f1"]
metric_lbls = ["Accuracy", "AUC", "Precision", "Recall", "F1"]
colors_m    = [P["blue"], P["accent"], P["green"], P["orange"], P["yellow"]]
x = np.arange(len(model_names)); w = 0.15

fig5, ax5 = plt.subplots(figsize=(13, 5), facecolor=P["bg"])
ax5.set_facecolor(P["card"])
for mi, (metric, lbl, col) in enumerate(zip(metrics, metric_lbls, colors_m)):
    vals = [results[n][metric] for n in model_names]
    bars = ax5.bar(x + mi*w - 2*w, vals, w, label=lbl,
                   color=col, edgecolor=P["border"], linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f"{v:.2f}", ha="center", va="bottom", fontsize=7,
                 color=P["text"], rotation=90)

thresh_labels = [f"{n}\n(t={results[n]['threshold']:.2f})" for n in model_names]
ax5.set_xticks(x); ax5.set_xticklabels(thresh_labels, fontsize=9)
ax5.set_ylim(0, 1.18); ax5.set_ylabel("Score")
ax5.set_title("Multi-Model Comparison — Real Held-Out Test Set (80/20 split, Youden threshold)",
              color=P["text"], fontweight="bold")
ax5.legend(facecolor=P["card"], edgecolor=P["border"], fontsize=9, loc="upper right")
ax5.grid(axis="y")

best_auc_vals = [results[n]["auc"] for n in model_names]
best_m_idx    = int(np.argmax(best_auc_vals))
ax5.annotate("Best AUC",
             xy=(best_m_idx + 1*w - 2*w, best_auc_vals[best_m_idx]+0.06),
             color=P["yellow"], fontsize=9, fontweight="bold", ha="center")
plt.tight_layout()
fig5.savefig("D:\\medical_1D\\new_output\\model_comparison.png",
             dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close(fig5)
print("Saved: csa_model_comparison.png")

# =============================================================================
# SAVE CLINICAL EXPLANATIONS
# =============================================================================
with open("D:\\medical_1D\\new_output\\clinical_explanations_csa.txt",
          "w", encoding="utf-8") as f:
    f.write("POPF PREDICTION -- CLINICAL EXPLANATIONS\n")
    f.write("Non-Traditional Feature Analysis\n")
    f.write("=" * 68 + "\n\n")
    f.write("GLOBAL FEATURE KNOWLEDGE BASE\n")
    f.write("-" * 68 + "\n\n")
    for fname, info in CLINICAL_KNOWLEDGE.items():
        if fname in final_features:
            fi = final_features.index(fname)
            ms = sv_popf[:, fi].mean()
            f.write(f"FEATURE: {info['display']}\n")
            f.write(f"  Normal range   : {info['normal_range']}\n")
            f.write(f"  Unit           : {info['unit']}\n")
            f.write(f"  Mean SHAP      : {ms:+.4f}\n")
            f.write(f"  Risk direction : {info['high_risk_dir'] if ms>0 else info['low_risk_dir']}\n")
            f.write(f"  Why it matters : {info['why_it_matters']}\n")
            f.write(f"  Doctor note    : {info['doctor_note']}\n\n")

    f.write("\n" + "=" * 68 + "\n")
    f.write("PER-PATIENT EXPLANATIONS (training set)\n")
    f.write("=" * 68 + "\n")
    for idx in range(len(y_tr)):
        f.write(clinical_explain(idx, X_tr, y_tr, sv_popf, shap_rf))

print("Saved: clinical_explanations_csa.txt")
import os
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

out_dir = r"D:\\medical_1D\\new_output"
os.makedirs(out_dir, exist_ok=True)

# ---------------------------------------------------------------------
# 1) CONFUSION MATRIX PLOTS
# ---------------------------------------------------------------------
for name, r in results.items():
    cm = confusion_matrix(y_te, r["y_pred"])

    fig, ax = plt.subplots(figsize=(5.5, 4.5), facecolor=P["bg"])
    ax.set_facecolor(P["card"])

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No POPF", "POPF"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")

    ax.set_title(f"Confusion Matrix — {name}", color=P["text"], fontweight="bold")
    ax.tick_params(colors=P["muted"])
    plt.tight_layout()

    fname = os.path.join(out_dir, f"cm_{name.lower().replace(' ', '_')}.png")
    fig.savefig(fname, dpi=150, bbox_inches="tight", facecolor=P["bg"])
    plt.close(fig)
    print(f"Saved: {fname}")

# ---------------------------------------------------------------------
# 2) FEATURE IMPORTANCE PLOT
#    - tree models: use feature_importances_
#    - non-tree models: fallback to permutation importance
# ---------------------------------------------------------------------
best_model_name = best_name
best_model = results[best_model_name]["model"]

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    importance_source = "Native feature importance"
else:
    print(f"{best_model_name} has no native feature_importances_. Using permutation importance...")
    perm = permutation_importance(
        best_model,
        X_te_scaled,
        y_te,
        n_repeats=30,
        random_state=SEED,
        scoring="roc_auc",
        n_jobs=-1
    )
    importances = perm.importances_mean
    importance_source = "Permutation importance"

imp_df = pd.DataFrame({
    "Feature": final_features,
    "Importance": importances
}).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(12, max(7, len(final_features) * 0.45 + 2)), facecolor=P["bg"])
ax.set_facecolor(P["card"])

bars = ax.barh(
    imp_df["Feature"],
    imp_df["Importance"],
    color=P["accent"],
    edgecolor=P["border"],
    height=0.65
)

for bar, val in zip(bars, imp_df["Importance"]):
    ax.text(
        bar.get_width() + 0.001,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}",
        va="center",
        fontsize=8,
        color=P["muted"]
    )

ax.set_xlabel("Importance")
ax.set_title(f"Feature Importance — {best_model_name} ({importance_source})_csa", color=P["text"], fontweight="bold")
ax.grid(axis="x")
plt.tight_layout()

fname = os.path.join(out_dir, f"feature_importance_{best_model_name.lower().replace(' ', '_')}_csa.png")
fig.savefig(fname, dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close(fig)
print(f"Saved: {fname}")

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "="*55)
print("IMPROVEMENTS APPLIED")
print("  1. KNN Imputer (k=5, distance-weighted) instead of median/mode")
print("  2. TARGET_N raised 200 → 300 (more augmented samples)")
print("  3. csa fitness: Accuracy → AUC  (more discriminative)")
print("  4. csa wasps/iters: 10/12 → 15/20 (Stage1), 8/10 → 12/15 (Stage2)")
print("  5. BorderlineSMOTE instead of standard SMOTE")
print("  6. RobustScaler on continuous features (critical for KNN)")
print("  7. Threshold: F1-only → Youden's J (sensitivity + specificity - 1)")
print("  8. RandomizedSearchCV n_iter: 60 → 100 + wider hyperparameter grids")
print("  9. RepeatedStratifiedKFold (5×3) for threshold optimisation")
print("="*55)
print("DONE")
print(f"  Real train / test split : {len(X_train_r)} / {len(X_test_r)}")
print(f"  Augmented training size : {len(X_aug)}")
print(f"  Final features          : {len(final_features)}")
print(f"\n  -- Test-set results (real held-out patients, Youden threshold) --")
for name, r in results.items():
    mark = "  [BEST AUC]" if name == best_name else ""
    print(f"  {name:<18}  Acc={r['accuracy']:.3f}  AUC={r['auc']:.3f}  "
          f"Rec={r['recall']:.3f}  F1={r['f1']:.3f}  thresh={r['threshold']:.2f}{mark}")
print()

Raw: (76, 54)

── KNN Imputation (k=5) for missing values ──
Binary columns  (mode-imputed) : 7
Continuous cols (KNN-imputed)  : 20

Original: 74 patients | POPF: 43  No-POPF: 31

STEP 2 — 80/20 TRAIN/TEST SPLIT (before augmentation)
Train (real): 59 patients | POPF=34  No-POPF=25
Test  (real): 15  patients | POPF=9  No-POPF=6
  Test set = real patients only, never used during augmentation or tuning

── Linear interpolation augmentation (train only) ──
Real train : 59  |  Augmented train : 300  |  Test (real) : 15
  Binary column integrity verified

STAGE 1 — csa on all features  [n_wasps=15, n_iter=20, AUC fitness]
  [CSA Stage-1] Init best=0.9998 features=13
  [CSA Stage-1] Iter 1/20  best=0.9998 features=13


KeyboardInterrupt: 

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import re, numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import shap

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import KNNImputer
from sklearn.model_selection import (StratifiedKFold,
                                     train_test_split, RandomizedSearchCV,
                                     cross_val_predict)
from sklearn.metrics import (classification_report, accuracy_score,
                              roc_auc_score, confusion_matrix,
                              precision_score, recall_score, f1_score)
from imblearn.over_sampling import BorderlineSMOTE
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("⚠ XGBoost not installed — skipping XGBoost model")

SEED = 42
np.random.seed(SEED)

# ── Palette ──────────────────────────────────────────────────────────────────
P = {"bg":"#0F1117","card":"#1A1D27","accent":"#7C6AF7","green":"#4ADE80",
     "red":"#F87171","text":"#E8E8F0","muted":"#8B8FA8","border":"#2A2D3E",
     "orange":"#FB923C","blue":"#60A5FA","yellow":"#FCD34D"}

plt.rcParams.update({
    "figure.facecolor":P["bg"],"axes.facecolor":P["card"],
    "axes.edgecolor":P["border"],"axes.labelcolor":P["text"],
    "xtick.color":P["muted"],"ytick.color":P["muted"],"text.color":P["text"],
    "grid.color":P["border"],"grid.linestyle":"--","grid.alpha":0.4,
    "font.family":"sans-serif","axes.titlesize":11,"axes.labelsize":9,
})

# ── Non-traditional features (paper novelty) ─────────────────────────────────
NON_TRADITIONAL = {
    "hb"             : "Haemoglobin (g/dL)",
    "albumin "       : "Serum Albumin (g/dL)",
    "ALP"            : "Alkaline Phosphatase (U/L)",
    "HBA1C"          : "HbA1c (%)",
    "bill "          : "Serum Bilirubin (mg/dL)",
    "BLOOD LOSS "    : "Intra-op Blood Loss (mL)",
    "CBD DIAMETER"   : "CBD Diameter (mm)",
    "ca 19-9"        : "CA 19-9 (U/mL)",
    "Cea"            : "CEA (ng/mL)",
    "fever "         : "Pre-op Fever",
    "RECENT DIABETES": "Recent Diabetes",
    "comorbidity"    : "Comorbidity",
    "neoadjuvant "   : "Neoadjuvant Treatment",
    "ercp stenting " : "ERCP Stenting",
    "jaundice"       : "Obstructive Jaundice",
    "pain "          : "Pre-op Pain",
    "weight loss "   : "Weight Loss",
    "bleeding "      : "Pre-op Bleeding",
    "Age "           : "Patient Age",
    "Sex"            : "Patient Sex",
}

# =============================================================================
# STEP 1 — LOAD & CLEAN
# =============================================================================
df = pd.read_excel("D:\\medical_1D\\1D data 2.xlsx")
print(f"Raw: {df.shape}")

DROP = ["DGE","PPH","SSI","CLAVIEN DINDO","BILE LAK","HOSPITAL STAY ",
        "READMISION","RESURGERY","mortality",
        "s no ","Name ","Address","CR Number ",
        "Unnamed: 5","Unnamed: 6","Unnamed: 7",
        "date of admisson ","date of sx ","date of discharge ","date of surgery",
        "cect","eus ","periampullary carcinoma ","surgery","CHROMAGANIN "]
df.drop(columns=[c for c in DROP if c in df.columns], inplace=True)

df.dropna(subset=["POPF"], inplace=True)
df["POPF"] = (df["POPF"] != 0).astype(int)

def auto_encode(col, series):
    name = col.strip().lower()
    if series.dtype in [np.float64, np.int64, float, int]:
        num = pd.to_numeric(series, errors="coerce")
        return num if num.notna().mean() >= 0.4 else None
    s = series.astype(str).str.strip()
    if name == "lap open": return None
    if name in ("fever ", "neoadjuvant "):
        def _b(v):
            sv = str(v).strip()
            if sv in ("-","9","nan",""): return np.nan
            try:   return 1.0 if float(sv) > 0 else 0.0
            except: return np.nan
        return series.apply(_b)
    if name == "ercp stenting ":
        return series.apply(lambda v: 1.0 if str(v).strip().startswith("1")
                            else (0.0 if str(v).strip() == "0" else np.nan))
    if name == "comorbidity":
        return series.apply(lambda v: 0.0 if str(v).strip() in ("0","nan","")
                            else (0.0 if str(v).strip() == "0" else 1.0))
    if name == "bill ":
        return pd.to_numeric(s.str.replace(r"[Oo]","0",regex=True), errors="coerce")
    if name == "cea":
        return pd.to_numeric(s.str.replace("`",""), errors="coerce")
    if name == "tumor size":
        return series.apply(lambda v: max(
            (float(n) for n in re.findall(r"[\d.]+", str(v))), default=np.nan))
    num = pd.to_numeric(s.str.replace("`","").str.replace(",","."), errors="coerce")
    return num if num.notna().mean() >= 0.4 else None

y   = df["POPF"].astype(int)
raw = df.drop(columns=["POPF"])
keep = {}
for col in raw.columns:
    res = auto_encode(col, raw[col])
    if res is not None and res.notna().mean() >= 0.20 and res.nunique() > 1:
        keep[col] = res.astype(float)

X_df = pd.DataFrame(keep)
feature_names = list(X_df.columns)

def is_binary_col(series):
    vals = set(series.dropna().unique())
    return vals.issubset({0.0, 1.0})

binary_cols     = [c for c in X_df.columns if is_binary_col(X_df[c])]
continuous_cols = [c for c in X_df.columns if c not in binary_cols]

# ── IMPROVEMENT 1: KNN Imputer for smarter missing value filling ──────────────
print("\n── KNN Imputation (k=5) for missing values ──")
knn_imp = KNNImputer(n_neighbors=5, weights="distance")

# Impute continuous cols with KNN, binary with mode (preserves 0/1 integrity)
X_cont = X_df[continuous_cols].values
X_cont_imputed = knn_imp.fit_transform(X_cont)
for i, col in enumerate(continuous_cols):
    X_df[col] = X_cont_imputed[:, i]
for col in binary_cols:
    X_df[col].fillna(X_df[col].mode()[0], inplace=True)

print(f"Binary columns  (mode-imputed) : {len(binary_cols)}")
print(f"Continuous cols (KNN-imputed)  : {len(continuous_cols)}")

X_np = X_df.values.copy()
y_np = y.values.copy()
binary_idx     = [feature_names.index(c) for c in binary_cols]
continuous_idx = [feature_names.index(c) for c in continuous_cols]
print(f"\nOriginal: {len(y_np)} patients | POPF: {y_np.sum()}  No-POPF: {(y_np==0).sum()}")

# =============================================================================
# STEP 2 — 80/20 TRAIN/TEST SPLIT (BEFORE AUGMENTATION — no leakage)
# =============================================================================
print("\n" + "="*55)
print("STEP 2 — 80/20 TRAIN/TEST SPLIT (before augmentation)")
print("="*55)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_np, y_np, test_size=0.20, stratify=y_np, random_state=SEED)
print(f"Train (real): {len(y_train_r)} patients | POPF={y_train_r.sum()}  No-POPF={(y_train_r==0).sum()}")
print(f"Test  (real): {len(y_test_r)}  patients | POPF={y_test_r.sum()}  No-POPF={(y_test_r==0).sum()}")
print("  Test set = real patients only, never used during augmentation or tuning")

# =============================================================================
# STEP 3 — BINARY-SAFE INTERPOLATION AUGMENTATION (training set only)
# ── IMPROVEMENT 2: TARGET_N raised 200→300 for more robust training ──────────
# =============================================================================
print("\n── Linear interpolation augmentation (train only) ──")

TARGET_N = 300                      # ← raised from 200
cont_idx = [i for i in range(X_np.shape[1]) if i not in binary_idx]
rng_aug  = np.random.default_rng(SEED)

def linear_interpolate_class(X_class, n_needed, binary_idx, cont_idx, rng):
    n_real, n_feat = X_class.shape
    X_syn = np.empty((n_needed, n_feat), dtype=float)
    for s in range(n_needed):
        a, b = rng.choice(n_real, size=2, replace=(n_real < 2))
        lam  = rng.uniform(0.0, 1.0)
        X_syn[s, cont_idx]   = lam * X_class[a, cont_idx] + (1.0-lam) * X_class[b, cont_idx]
        X_syn[s, binary_idx] = np.where(lam >= 0.5,
                                         X_class[a, binary_idx],
                                         X_class[b, binary_idx])
    return X_syn

idx_pos = np.where(y_train_r == 1)[0]
idx_neg = np.where(y_train_r == 0)[0]
X_pos, X_neg = X_train_r[idx_pos], X_train_r[idx_neg]
half      = TARGET_N // 2
n_syn_pos = max(0, half - len(X_pos))
n_syn_neg = max(0, half - len(X_neg))
X_syn_pos = linear_interpolate_class(X_pos, n_syn_pos, binary_idx, cont_idx, rng_aug)
X_syn_neg = linear_interpolate_class(X_neg, n_syn_neg, binary_idx, cont_idx, rng_aug)

X_aug = np.vstack([X_train_r, X_syn_pos, X_syn_neg])
y_aug = np.concatenate([y_train_r,
                        np.ones(n_syn_pos, dtype=int),
                        np.zeros(n_syn_neg, dtype=int)])

print(f"Real train : {len(X_train_r)}  |  Augmented train : {len(X_aug)}  |  Test (real) : {len(X_test_r)}")

for idx in binary_idx:
    vals = set(np.unique(X_aug[:, idx]))
    assert vals.issubset({0.0, 1.0}), f"Binary violated: {feature_names[idx]}"
print("  Binary column integrity verified")

# =============================================================================
def abc_optimizer(X_data, y_data, n_bees, n_iter,
                  limit=5, min_feat=3, label="ABC"):

    n_feat = X_data.shape[1]
    rng = np.random.default_rng(SEED)

    # ── Fitness (same AUC-based as before) ───────────────────────────
    def fitness(mask):
        if mask.sum() == 0:
            return 0.0

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
        aucs = []

        for tr, te in skf.split(X_data[:, mask], y_data):
            Xtr, Xte = X_data[tr][:, mask], X_data[te][:, mask]
            ytr, yte = y_data[tr], y_data[te]

            k = min(4, int(ytr.sum()) - 1)
            if k < 1:
                continue

            try:
                Xs, ys = BorderlineSMOTE(random_state=SEED, k_neighbors=k).fit_resample(Xtr, ytr)
            except:
                Xs, ys = Xtr, ytr

            model = RandomForestClassifier(
                n_estimators=50,
                max_depth=5,
                class_weight="balanced",
                random_state=SEED
            )
            model.fit(Xs, ys)

            prob = model.predict_proba(Xte)[:, 1]
            if len(np.unique(yte)) > 1:
                aucs.append(roc_auc_score(yte, prob))

        return float(np.mean(aucs)) if aucs else 0.0

    # ── Ensure minimum features ──────────────────────────────────────
    def enforce_min(mask):
        if mask.sum() < min_feat:
            off = np.where(mask == 0)[0]
            need = min_feat - int(mask.sum())
            mask[rng.choice(off, min(need, len(off)), replace=False)] = 1
        return mask

    # ── Initialize food sources (solutions) ──────────────────────────
    food_sources = rng.integers(0, 2, (n_bees, n_feat)).astype(float)
    food_sources = np.array([enforce_min(f) for f in food_sources])

    fitness_vals = np.array([fitness(f.astype(bool)) for f in food_sources])
    trials = np.zeros(n_bees)

    best_idx = np.argmax(fitness_vals)
    g_best = food_sources[best_idx].copy()
    g_score = fitness_vals[best_idx]

    history = [g_score]
    print(f"  [{label}] Init best={g_score:.4f} features={int(g_best.sum())}")

    # ── ABC main loop ────────────────────────────────────────────────
    for it in range(n_iter):

        # ── Employed Bee Phase ───────────────────────────────────────
        for i in range(n_bees):
            k = rng.integers(0, n_bees)
            while k == i:
                k = rng.integers(0, n_bees)

            phi = rng.uniform(-1, 1, n_feat)
            new_sol = food_sources[i] + phi * (food_sources[i] - food_sources[k])

            new_sol = (new_sol > 0.5).astype(float)
            new_sol = enforce_min(new_sol)

            new_fit = fitness(new_sol.astype(bool))

            if new_fit > fitness_vals[i]:
                food_sources[i] = new_sol
                fitness_vals[i] = new_fit
                trials[i] = 0
            else:
                trials[i] += 1

        # ── Onlooker Bee Phase ───────────────────────────────────────
        probs = fitness_vals / (fitness_vals.sum() + 1e-9)

        for _ in range(n_bees):
            i = rng.choice(n_bees, p=probs)

            k = rng.integers(0, n_bees)
            while k == i:
                k = rng.integers(0, n_bees)

            phi = rng.uniform(-1, 1, n_feat)
            new_sol = food_sources[i] + phi * (food_sources[i] - food_sources[k])

            new_sol = (new_sol > 0.5).astype(float)
            new_sol = enforce_min(new_sol)

            new_fit = fitness(new_sol.astype(bool))

            if new_fit > fitness_vals[i]:
                food_sources[i] = new_sol
                fitness_vals[i] = new_fit
                trials[i] = 0
            else:
                trials[i] += 1

        # ── Scout Bee Phase ──────────────────────────────────────────
        for i in range(n_bees):
            if trials[i] > limit:
                food_sources[i] = rng.integers(0, 2, n_feat)
                food_sources[i] = enforce_min(food_sources[i])
                fitness_vals[i] = fitness(food_sources[i].astype(bool))
                trials[i] = 0

        # ── Update global best ───────────────────────────────────────
        best_idx = np.argmax(fitness_vals)
        if fitness_vals[best_idx] > g_score:
            g_best = food_sources[best_idx].copy()
            g_score = fitness_vals[best_idx]

        history.append(g_score)

        if (it + 1) % 5 == 0 or it == 0:
            print(f"  [{label}] Iter {it+1}/{n_iter} best={g_score:.4f} features={int(g_best.sum())}")

    print(f"  [{label}] Done best={g_score:.4f} features={int(g_best.sum())}")
    return g_best.astype(bool), g_score, history

print("\n" + "="*55)
print("STAGE 1 — abc on all features  [n_bees=15, n_iter=20, AUC fitness]")
print("="*55)
mask_A, score_A, hist_A = abc_optimizer(
    X_aug, y_aug,
    n_bees=15,
    n_iter=20,
    limit=5,
    min_feat=4,
    label="ABC Stage-1"
)
set_A = [feature_names[i] for i, m in enumerate(mask_A) if m]
set_B = [feature_names[i] for i, m in enumerate(mask_A) if not m]
print(f"\n  Set A (important)  : {len(set_A)}  ->  {set_A}")
print(f"  Set B (unimportant): {len(set_B)}  ->  {set_B}")

print("\n" + "="*55)
print("STAGE 2 — abc on unimportant Set B  [n_bees=12, n_iter=15]")
print("="*55)
idx_B = [i for i, m in enumerate(mask_A) if not m]

mask_C, score_C, hist_C = abc_optimizer(
    X_aug[:, idx_B], y_aug,
    n_bees=12,
    n_iter=15,
    limit=5,
    min_feat=2,
    label="ABC Stage-2"
)
set_C = [set_B[i] for i, m in enumerate(mask_C) if m]
print(f"\n  Set C (rescued): {len(set_C)}  ->  {set_C}")

final_features = list(dict.fromkeys(set_A + set_C))
print(f"\nFinal features (A + C): {len(final_features)}")
for f in final_features:
    tag = "  [NON-TRAD]" if f in NON_TRADITIONAL else ""
    src = "A" if f in set_A else "C"
    print(f"  [{src}]  {f}{tag}")

# =============================================================================
# STEP 5 — MULTI-MODEL TRAINING + DEEP HYPERPARAMETER TUNING
# ── IMPROVEMENT 4: RobustScaler pipelines, BorderlineSMOTE, Youden threshold,
#                   n_iter=100, wider grids, RepeatedStratifiedKFold ──────────
# =============================================================================
feat_idx = [feature_names.index(f) for f in final_features]
X_tr     = X_aug[:, feat_idx].copy()
y_tr     = y_aug.copy()
X_te     = X_test_r[:, feat_idx].copy()
y_te     = y_test_r.copy()

# ── Feature scaling on continuous columns only ────────────────────────────────
final_cont_mask = np.array([f not in binary_cols for f in final_features])
scaler = RobustScaler()

def scale_data(X_train, X_test, cont_mask):
    """Fit scaler on train continuous cols, transform both — no leakage."""
    X_tr_s = X_train.copy()
    X_te_s = X_test.copy()
    if cont_mask.any():
        X_tr_s[:, cont_mask] = scaler.fit_transform(X_train[:, cont_mask])
        X_te_s[:, cont_mask] = scaler.transform(X_test[:, cont_mask])
    return X_tr_s, X_te_s

X_tr_scaled, X_te_scaled = scale_data(X_tr, X_te, final_cont_mask)

# ── BorderlineSMOTE on augmented training set ─────────────────────────────────
k_sm = min(4, int(y_tr.sum()) - 1)
try:
    Xs_tr, ys_tr = BorderlineSMOTE(random_state=SEED, k_neighbors=k_sm).fit_resample(
        X_tr_scaled, y_tr)
except Exception:
    from imblearn.over_sampling import SMOTE
    Xs_tr, ys_tr = SMOTE(random_state=SEED, k_neighbors=k_sm).fit_resample(
        X_tr_scaled, y_tr)

neg_count = int((ys_tr == 0).sum())
pos_count = int((ys_tr == 1).sum())
spw       = round(neg_count / max(pos_count, 1), 2)
print(f"\nBorderlineSMOTE output -> neg={neg_count}  pos={pos_count}  scale_pos_weight={spw}")

print("\n" + "="*55)
print("STEP 5 — MULTI-MODEL TRAINING + DEEP HYPERPARAMETER TUNING")
print(f"  Train : {len(Xs_tr)} (BorderlineSMOTE on augmented) | Test : {len(X_te_scaled)} (real)")
print(f"  Scaling: RobustScaler on {final_cont_mask.sum()} continuous features")
print("="*55)

# ── IMPROVEMENT 5: Youden's J threshold (sensitivity + specificity − 1) ──────
def best_threshold_youden(model, X, y):
    """
    Find threshold maximising Youden's J = Sensitivity + Specificity − 1.
    More robust than F1-only for imbalanced medical data.
    Note: cross_val_predict requires partitions so we use StratifiedKFold(10).
    """
    skf      = StratifiedKFold(n_splits=10, shuffle=True, random_state=SEED)
    cv_prob  = cross_val_predict(model, X, y, cv=skf, method="predict_proba")[:, 1]
    thresholds = np.arange(0.20, 0.81, 0.02)
    best_t, best_j = 0.5, -1.0
    for t in thresholds:
        yp  = (cv_prob >= t).astype(int)
        rec = recall_score(y, yp, zero_division=0)                    # sensitivity
        tn  = confusion_matrix(y, yp).ravel()
        # specificity = TN / (TN + FP)
        cm  = confusion_matrix(y, yp)
        if cm.shape == (2, 2):
            spec = cm[0, 0] / max(cm[0, 0] + cm[0, 1], 1)
        else:
            spec = 0.0
        j = rec + spec - 1.0
        if j > best_j:
            best_j = j
            best_t = t
    return float(best_t)

# ── Model definitions ─────────────────────────────────────────────────────────
MODELS = {}

# 1. Random Forest — wider grid, more estimators
MODELS["Random Forest"] = (
    RandomForestClassifier(random_state=SEED),
    {
        "n_estimators": [500, 700, 1000],
        "max_depth": [6, 8, 10, 12, None],
        "min_samples_leaf"  : [1, 2],
        "min_samples_split" : [2,3],
        "max_features"      : ["sqrt",0.7,None],
        "class_weight"      : ["balanced", "balanced_subsample",
                               {0:1,1:2}, {0:1,1:3}, {0:1,1:4}],
        "max_samples"       : [0.7, 0.8, 0.9, None],
    }
)

# 2. Decision Tree
MODELS["Decision Tree"] = (
    DecisionTreeClassifier(random_state=SEED),
    {
        "max_depth"         : [3, 4, 5, 6, 7, 8, None],
        "min_samples_leaf"  : [1, 2, 3, 5],
        "min_samples_split" : [2, 4, 6],
        "criterion"         : ["gini", "entropy"],
        "class_weight"      : ["balanced", {0:1,1:2}, {0:1,1:3}, {0:1,1:4}],
        "ccp_alpha"         : [0.0, 0.002, 0.005, 0.01, 0.02, 0.05],
        "splitter"          : ["best", "random"],
    }
)

# 3. XGBoost — deeper regularisation grid
if HAS_XGB:
    MODELS["XGBoost"] = (
        XGBClassifier(random_state=SEED, eval_metric="logloss", verbosity=0),
        {
            "n_estimators"     : [100, 200, 300, 500, 700],
            "max_depth"        : [3, 4, 5, 6, 7],
            "learning_rate"    : [0.005, 0.01, 0.05, 0.1, 0.2],
            "subsample"        : [0.6, 0.7, 0.8, 0.9, 1.0],
            "colsample_bytree" : [0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
            "min_child_weight" : [1, 2, 3, 5, 7],
            "gamma"            : [0, 0.05, 0.1, 0.2, 0.3, 0.5],
            "reg_alpha"        : [0, 0.01, 0.05, 0.1, 0.5, 1.0],
            "reg_lambda"       : [0.5, 1.0, 1.5, 2.0, 3.0],
            "scale_pos_weight" : [1.0, 1.5, spw, spw*1.5, spw*2.0],
            "max_delta_step"   : [0, 1, 5],   # helps with imbalanced data
        }
    )

# 4. KNN — uses scaled data, much wider metric/neighbour grid
MODELS["KNN"] = (
    KNeighborsClassifier(),
    {
        "n_neighbors": [15, 21,25],
        "weights"    : ["distance"],
        "metric"     : ["euclidean", "manhattan"],
        "p"          : [1, 2, 3],
        "leaf_size"  : [10, 20, 30, 40, 50],
        "algorithm"  : ["auto", "ball_tree", "kd_tree"],
    }
)

# ── Train + tune + threshold-optimise each model ──────────────────────────────
results = {}
best_rf_model = None

print(f"\n{'Model':<18} {'Best AUC(CV)':>12} {'Thresh':>7} "
      f"{'Test Acc':>9} {'Test AUC':>9} {'Prec':>7} {'Rec':>7} {'F1':>7}")
print("─" * 82)

# ── IMPROVEMENT 6: n_iter=100 (from 60) for more thorough search ──────────────
SEARCH_ITER = 100

for name, (base_model, param_grid) in MODELS.items():
    search = RandomizedSearchCV(
        base_model, param_grid,
        n_iter=SEARCH_ITER, cv=5, scoring="roc_auc",
        n_jobs=-1, refit=True, random_state=SEED
    )
    search.fit(Xs_tr, ys_tr)
    best_model = search.best_estimator_

    # Youden's J threshold on training CV
    thresh = best_threshold_youden(best_model, Xs_tr, ys_tr)

    # Evaluate on real held-out test set (scaled)
    ypr = best_model.predict_proba(X_te_scaled)[:, 1]
    yp  = (ypr >= thresh).astype(int)

    acc  = accuracy_score(y_te, yp)
    auc  = roc_auc_score(y_te, ypr) if len(np.unique(y_te)) > 1 else np.nan
    prec = precision_score(y_te, yp, zero_division=0)
    rec  = recall_score(y_te, yp, zero_division=0)
    f1   = f1_score(y_te, yp, zero_division=0)

    results[name] = {
        "model"       : best_model,
        "best_params" : search.best_params_,
        "cv_auc"      : search.best_score_,
        "threshold"   : thresh,
        "y_pred"      : yp,
        "y_prob"      : ypr,
        "accuracy"    : acc,
        "auc"         : auc,
        "precision"   : prec,
        "recall"      : rec,
        "f1"          : f1,
    }

    print(f"{name:<18} {search.best_score_:>12.3f} {thresh:>7.2f} "
          f"{acc:>9.3f} {auc:>9.3f} {prec:>7.3f} {rec:>7.3f} {f1:>7.3f}")

    if name == "Random Forest":
        best_rf_model = best_model

# ── Detailed reports ──────────────────────────────────────────────────────────
print("\n" + "="*55)
print("DETAILED CLASSIFICATION REPORTS (real held-out test set)")
print("="*55)
for name, r in results.items():
    print(f"\n-- {name} --")
    print(f"  Best params : {r['best_params']}")
    print(f"  Threshold   : {r['threshold']:.2f}  (Youden's J on training CV)")
    print(classification_report(y_te, r["y_pred"], target_names=["No POPF","POPF"]))

best_name = max(results, key=lambda n: results[n]["auc"])
print(f"\n  Best model by Test AUC: {best_name}  (AUC={results[best_name]['auc']:.3f})")

# =============================================================================
# STEP 6 — SHAP + CLINICAL EXPLAINER
# (SHAP fitted on UNscaled X_tr so feature values remain interpretable)
# =============================================================================
CLINICAL_KNOWLEDGE = {
    "hb": {
        "display"    : "Haemoglobin",
        "unit"       : "g/dL",
        "low_thresh" : 10.0, "high_thresh": None,
        "low_status" : "Anaemic", "high_status": "Normal",
        "normal_range": "12-16 g/dL (F), 13-17 g/dL (M)",
        "low_risk_dir": "increases POPF risk", "high_risk_dir": "decreases POPF risk",
        "why_it_matters": (
            "Low haemoglobin indicates pre-operative anaemia and tissue oxygen deprivation. "
            "Poorly oxygenated pancreatic tissue heals slowly, weakening the anastomosis "
            "and making a pancreatic fistula more likely."),
        "doctor_note": "Doctors routinely check Hb but rarely link it to POPF risk.",
    },
    "albumin ": {
        "display"    : "Serum Albumin",
        "unit"       : "g/dL",
        "low_thresh" : 3.5, "high_thresh": None,
        "low_status" : "Hypoalbuminaemia", "high_status": "Normal",
        "normal_range": "3.5-5.0 g/dL",
        "low_risk_dir": "increases POPF risk", "high_risk_dir": "decreases POPF risk",
        "why_it_matters": (
            "Albumin is the primary marker of nutritional status. Hypoalbuminaemia "
            "impairs wound healing, reduces anastomotic tensile strength, and increases "
            "susceptibility to post-operative leaks."),
        "doctor_note": "Serum albumin is checked pre-op but not integrated into POPF risk scoring.",
    },
    "ALP": {
        "display"    : "Alkaline Phosphatase (ALP)",
        "unit"       : "U/L",
        "low_thresh" : None, "high_thresh": 120.0,
        "low_status" : "Normal", "high_status": "Elevated",
        "normal_range": "44-147 U/L",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Elevated ALP reflects biliary obstruction and hepatic stress. Biliary "
            "hypertension raises intra-ductal pancreatic pressure, compromising "
            "anastomotic integrity and increasing fistula risk."),
        "doctor_note": "ALP is ordered as part of a liver panel but not used in POPF prediction.",
    },
    "HBA1C": {
        "display"    : "HbA1c (Glycated Haemoglobin)",
        "unit"       : "%",
        "low_thresh" : None, "high_thresh": 6.5,
        "low_status" : "Normal glycaemic control", "high_status": "Poor glycaemic control",
        "normal_range": "< 5.7% normal, 5.7-6.4% pre-diabetic, >= 6.5% diabetic",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Chronically elevated blood sugar causes microvascular damage and impairs "
            "neutrophil function, delaying anastomotic healing and increasing "
            "vulnerability to post-operative complications."),
        "doctor_note": "HbA1c is checked for diabetics but overlooked as a surgical risk factor.",
    },
    "bill ": {
        "display"    : "Serum Bilirubin",
        "unit"       : "mg/dL",
        "low_thresh" : None, "high_thresh": 2.0,
        "low_status" : "Normal", "high_status": "Hyperbilirubinaemia",
        "normal_range": "0.2-1.2 mg/dL",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Pre-operative jaundice and elevated bilirubin cause Kupffer-cell "
            "dysfunction and endotoxaemia, impairing the immune response and "
            "increasing post-operative fistula complications."),
        "doctor_note": "Bilirubin is well-monitored for biliary disease but not in POPF models.",
    },
    "BLOOD LOSS ": {
        "display"    : "Intra-operative Blood Loss",
        "unit"       : "mL",
        "low_thresh" : None, "high_thresh": 500.0,
        "low_status" : "Acceptable", "high_status": "Excessive",
        "normal_range": "< 500 mL acceptable for Whipple",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Excessive intra-operative haemorrhage triggers coagulopathy and "
            "systemic inflammatory response, impairing anastomotic integrity "
            "and increasing the likelihood of fistula formation."),
        "doctor_note": "Blood loss is recorded but not used quantitatively in POPF prediction.",
    },
    "CBD DIAMETER": {
        "display"    : "Common Bile Duct Diameter",
        "unit"       : "mm",
        "low_thresh" : None, "high_thresh": 8.0,
        "low_status" : "Normal", "high_status": "Dilated",
        "normal_range": "< 6-8 mm",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "A dilated CBD indicates chronic biliary obstruction, leading to hepatic "
            "reserve impairment and elevated intra-ductal pressure, compromising "
            "anastomotic healing after pancreaticoduodenectomy."),
        "doctor_note": "CBD diameter is measured on imaging but rarely used as a POPF predictor.",
    },
    "ca 19-9": {
        "display"    : "CA 19-9 Tumour Marker",
        "unit"       : "U/mL",
        "low_thresh" : None, "high_thresh": 37.0,
        "low_status" : "Normal", "high_status": "Elevated",
        "normal_range": "< 37 U/mL",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Elevated CA 19-9 reflects tumour burden and associated inflammatory "
            "milieu, which may compromise tissue quality and anastomotic healing."),
        "doctor_note": "CA 19-9 is used for diagnosis but not for POPF prediction.",
    },
    "weight loss ": {
        "display"    : "Pre-operative Weight Loss",
        "unit"       : "(binary: 1=yes)",
        "low_thresh" : None, "high_thresh": None,
        "low_status" : "No weight loss", "high_status": "Significant weight loss",
        "normal_range": "Absent",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Pre-operative weight loss is a surrogate for cachexia and protein-energy "
            "malnutrition. Malnourished patients have impaired wound healing and higher "
            "anastomotic complication rates."),
        "doctor_note": "Weight loss is documented clinically but not formally scored for POPF.",
    },
    "fever ": {
        "display"    : "Pre-operative Fever",
        "unit"       : "(binary: 1=yes)",
        "low_thresh" : None, "high_thresh": None,
        "low_status" : "Afebrile", "high_status": "Febrile",
        "normal_range": "Absent",
        "low_risk_dir": "decreases POPF risk", "high_risk_dir": "increases POPF risk",
        "why_it_matters": (
            "Pre-operative fever indicates active infection or systemic inflammation. "
            "Operating in an inflammatory state predisposes to impaired anastomotic "
            "healing and higher fistula rates."),
        "doctor_note": "Fever is a general surgical risk factor but not in standard POPF scores.",
    },
}

def get_status(feat, value):
    info = CLINICAL_KNOWLEDGE.get(feat)
    if not info: return "--"
    if info["low_thresh"] and value < info["low_thresh"]:
        return f"{info['low_status']} (< {info['low_thresh']} {info['unit']})"
    if info["high_thresh"] and value > info["high_thresh"]:
        return f"{info['high_status']} (> {info['high_thresh']} {info['unit']})"
    return f"Normal range ({info['normal_range']})"

# Fit SHAP on unscaled training data (interpretable feature values)
print("\n-- Computing SHAP (Random Forest, unscaled X_tr) --")
shap_rf    = best_rf_model
# Re-fit on unscaled data for SHAP interpretability
shap_rf.fit(
    *BorderlineSMOTE(random_state=SEED, k_neighbors=min(4,int(y_tr.sum())-1)).fit_resample(
        X_tr, y_tr)
    if min(4,int(y_tr.sum())-1) >= 1 else (X_tr, y_tr)
)
X_df_shap  = pd.DataFrame(X_tr, columns=final_features)
explainer  = shap.TreeExplainer(shap_rf)
sv         = explainer(X_df_shap)
sv_popf    = sv.values[:, :, 1]


def clinical_explain(idx, X_data, y_data, sv_data, model):
    sv_  = sv_data[idx]
    fv   = X_data[idx]
    prob = model.predict_proba(X_data[idx:idx+1])[0, 1]
    pred = "POPF" if prob >= 0.5 else "No POPF"
    true = "POPF" if y_data[idx] == 1 else "No POPF"

    nt_rows, tr_rows = [], []
    for fi, fname in enumerate(final_features):
        if fname in CLINICAL_KNOWLEDGE or fname in NON_TRADITIONAL:
            nt_rows.append((fi, fname, sv_[fi], fv[fi]))
        else:
            tr_rows.append((fi, fname, sv_[fi], fv[fi]))

    nt_rows.sort(key=lambda x: abs(x[2]), reverse=True)
    tr_rows.sort(key=lambda x: abs(x[2]), reverse=True)

    sep = "=" * 68
    out = (f"\n{sep}\n"
           f"  PATIENT {idx+1}  |  Prediction: {pred}  |  Actual: {true}\n"
           f"  POPF Risk Probability: {prob*100:.1f}%\n"
           f"{sep}\n\n")

    out += "  NON-TRADITIONAL FEATURES\n"
    out += "  " + "-" * 64 + "\n"
    for fi, fname, shap_val, val in nt_rows:
        info    = CLINICAL_KNOWLEDGE.get(fname, {})
        display = info.get("display", NON_TRADITIONAL.get(fname, fname))
        unit    = info.get("unit", "")
        status  = get_status(fname, val)
        arrow   = "increases POPF risk" if shap_val > 0 else "decreases POPF risk"
        out += f"\n  * {display}\n"
        out += f"    Value  : {val:.2f} {unit}\n"
        out += f"    Status : {status}\n"
        out += f"    Effect : {arrow}  (SHAP = {shap_val:+.4f})\n"
        if fname in CLINICAL_KNOWLEDGE:
            why = info["why_it_matters"]
            words = why.split(); line = "      "
            for w in words:
                if len(line) + len(w) > 66: out += line + "\n"; line = "      "
                line += w + " "
            out += line.rstrip() + "\n"

    out += f"\n  TRADITIONAL FEATURES (top 5)\n"
    out += "  " + "-" * 64 + "\n"
    for fi, fname, shap_val, val in tr_rows[:5]:
        arrow = "higher POPF" if shap_val > 0 else "lower POPF"
        out  += f"  . {fname.strip():<25} = {val:>8.2f}   SHAP {shap_val:+.4f}  {arrow}\n"
    return out

popf_idx   = np.where(y_tr == 1)[0][0]
nopopf_idx = np.where(y_tr == 0)[0][0]
print(clinical_explain(popf_idx,   X_tr, y_tr, sv_popf, shap_rf))
print(clinical_explain(nopopf_idx, X_tr, y_tr, sv_popf, shap_rf))

# =============================================================================
# PLOTS (3 figures)
# =============================================================================

# ── Plot 1: YJO Convergence ───────────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(10, 4), facecolor=P["bg"])
ax2.set_facecolor(P["card"])
ax2.plot(hist_A, color=P["accent"], lw=2.5, marker="o", ms=5,
         label=f"Stage 1 — all {len(feature_names)} features")
ax2.plot(hist_C, color=P["orange"], lw=2.5, marker="s", ms=5, ls="--",
         label=f"Stage 2 — Set B ({len(set_B)} unimportant features)")
ax2.set_xlabel("Iteration"); ax2.set_ylabel("Best CV AUC")
ax2.set_title("abc Convergence — both stages (AUC fitness)", color=P["text"], fontweight="bold")
ax2.set_ylim(0.4, 1.05)
ax2.legend(facecolor=P["card"], edgecolor=P["border"], fontsize=9)
ax2.grid(True)
plt.tight_layout()
fig2.savefig("D:\\medical_1D\\new_output\\abc_convergence.png",
             dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close(fig2)
print("\nSaved: abc_convergence.png")

# ── Plot 2: Feature Importance ────────────────────────────────────────────────
imp_df = pd.DataFrame({
    "Feature"   : final_features,
    "Importance": shap_rf.feature_importances_,
    "Origin"    : ["A" if f in set_A else "C" for f in final_features],
    "Novel"     : [f in NON_TRADITIONAL for f in final_features],
}).sort_values("Importance", ascending=True)

fig3, ax3 = plt.subplots(figsize=(12, max(7, len(final_features)*0.45 + 2)),
                          facecolor=P["bg"])
ax3.set_facecolor(P["card"])
clrs = []
for _, row in imp_df.iterrows():
    if   row["Novel"] and row["Origin"] == "A": clrs.append(P["green"])
    elif row["Novel"] and row["Origin"] == "C": clrs.append(P["orange"])
    elif row["Origin"] == "A":                  clrs.append(P["blue"])
    else:                                        clrs.append(P["muted"])

bars = ax3.barh(imp_df["Feature"], imp_df["Importance"],
                color=clrs, edgecolor=P["border"], height=0.65, linewidth=0.4)
for bar, (_, row) in zip(bars, imp_df.iterrows()):
    lbl = f'{row["Importance"]:.3f}{"  [NT]" if row["Novel"] else ""}'
    ax3.text(bar.get_width()+0.001, bar.get_y()+bar.get_height()/2,
             lbl, va="center", fontsize=8, color=P["muted"])
ax3.set_xlabel("Mean RF Importance (Gini)")
ax3.set_title("CSA-selected Features — importance x origin x novelty",
              color=P["text"], fontweight="bold")
ax3.grid(axis="x")
ax3.legend(handles=[
    mpatches.Patch(color=P["green"],  label="Non-traditional + Set A"),
    mpatches.Patch(color=P["orange"], label="Non-traditional + Set C (rescued)"),
    mpatches.Patch(color=P["blue"],   label="Traditional + Set A"),
    mpatches.Patch(color=P["muted"],  label="Traditional + Set C"),
], facecolor=P["card"], edgecolor=P["border"], fontsize=8, loc="lower right")
plt.tight_layout()
fig3.savefig("D:\\medical_1D\\new_output\\feature_importance_abc.png",
             dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close(fig3)
print("Saved: feature_importance_abc.png")

# ── Plot 3: Multi-Model Comparison ────────────────────────────────────────────
model_names = list(results.keys())
metrics     = ["accuracy", "auc", "precision", "recall", "f1"]
metric_lbls = ["Accuracy", "AUC", "Precision", "Recall", "F1"]
colors_m    = [P["blue"], P["accent"], P["green"], P["orange"], P["yellow"]]
x = np.arange(len(model_names)); w = 0.15

fig5, ax5 = plt.subplots(figsize=(13, 5), facecolor=P["bg"])
ax5.set_facecolor(P["card"])
for mi, (metric, lbl, col) in enumerate(zip(metrics, metric_lbls, colors_m)):
    vals = [results[n][metric] for n in model_names]
    bars = ax5.bar(x + mi*w - 2*w, vals, w, label=lbl,
                   color=col, edgecolor=P["border"], linewidth=0.5)
    for bar, v in zip(bars, vals):
        ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f"{v:.2f}", ha="center", va="bottom", fontsize=7,
                 color=P["text"], rotation=90)

thresh_labels = [f"{n}\n(t={results[n]['threshold']:.2f})" for n in model_names]
ax5.set_xticks(x); ax5.set_xticklabels(thresh_labels, fontsize=9)
ax5.set_ylim(0, 1.18); ax5.set_ylabel("Score")
ax5.set_title("Multi-Model Comparison — Real Held-Out Test Set (80/20 split, Youden threshold)",
              color=P["text"], fontweight="bold")
ax5.legend(facecolor=P["card"], edgecolor=P["border"], fontsize=9, loc="upper right")
ax5.grid(axis="y")

best_auc_vals = [results[n]["auc"] for n in model_names]
best_m_idx    = int(np.argmax(best_auc_vals))
ax5.annotate("Best AUC",
             xy=(best_m_idx + 1*w - 2*w, best_auc_vals[best_m_idx]+0.06),
             color=P["yellow"], fontsize=9, fontweight="bold", ha="center")
plt.tight_layout()
fig5.savefig("D:\\medical_1D\\new_output\\model_comparison.png",
             dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close(fig5)
print("Saved: csa_model_comparison.png")

# =============================================================================
# SAVE CLINICAL EXPLANATIONS
# =============================================================================
with open("D:\\medical_1D\\new_output\\clinical_explanations_abc.txt",
          "w", encoding="utf-8") as f:
    f.write("POPF PREDICTION -- CLINICAL EXPLANATIONS\n")
    f.write("Non-Traditional Feature Analysis\n")
    f.write("=" * 68 + "\n\n")
    f.write("GLOBAL FEATURE KNOWLEDGE BASE\n")
    f.write("-" * 68 + "\n\n")
    for fname, info in CLINICAL_KNOWLEDGE.items():
        if fname in final_features:
            fi = final_features.index(fname)
            ms = sv_popf[:, fi].mean()
            f.write(f"FEATURE: {info['display']}\n")
            f.write(f"  Normal range   : {info['normal_range']}\n")
            f.write(f"  Unit           : {info['unit']}\n")
            f.write(f"  Mean SHAP      : {ms:+.4f}\n")
            f.write(f"  Risk direction : {info['high_risk_dir'] if ms>0 else info['low_risk_dir']}\n")
            f.write(f"  Why it matters : {info['why_it_matters']}\n")
            f.write(f"  Doctor note    : {info['doctor_note']}\n\n")

    f.write("\n" + "=" * 68 + "\n")
    f.write("PER-PATIENT EXPLANATIONS (training set)\n")
    f.write("=" * 68 + "\n")
    for idx in range(len(y_tr)):
        f.write(clinical_explain(idx, X_tr, y_tr, sv_popf, shap_rf))

print("Saved: clinical_explanations_abc.txt")
import os
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.inspection import permutation_importance

out_dir = r"D:\\medical_1D\\new_output"
os.makedirs(out_dir, exist_ok=True)

# ---------------------------------------------------------------------
# 1) CONFUSION MATRIX PLOTS
# ---------------------------------------------------------------------
for name, r in results.items():
    cm = confusion_matrix(y_te, r["y_pred"])

    fig, ax = plt.subplots(figsize=(5.5, 4.5), facecolor=P["bg"])
    ax.set_facecolor(P["card"])

    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No POPF", "POPF"])
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")

    ax.set_title(f"Confusion Matrix — {name}", color=P["text"], fontweight="bold")
    ax.tick_params(colors=P["muted"])
    plt.tight_layout()

    fname = os.path.join(out_dir, f"cm_{name.lower().replace(' ', '_')}.png")
    fig.savefig(fname, dpi=150, bbox_inches="tight", facecolor=P["bg"])
    plt.close(fig)
    print(f"Saved: {fname}")

# ---------------------------------------------------------------------
# 2) FEATURE IMPORTANCE PLOT
#    - tree models: use feature_importances_
#    - non-tree models: fallback to permutation importance
# ---------------------------------------------------------------------
best_model_name = best_name
best_model = results[best_model_name]["model"]

if hasattr(best_model, "feature_importances_"):
    importances = best_model.feature_importances_
    importance_source = "Native feature importance"
else:
    print(f"{best_model_name} has no native feature_importances_. Using permutation importance...")
    perm = permutation_importance(
        best_model,
        X_te_scaled,
        y_te,
        n_repeats=30,
        random_state=SEED,
        scoring="roc_auc",
        n_jobs=-1
    )
    importances = perm.importances_mean
    importance_source = "Permutation importance"

imp_df = pd.DataFrame({
    "Feature": final_features,
    "Importance": importances
}).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(12, max(7, len(final_features) * 0.45 + 2)), facecolor=P["bg"])
ax.set_facecolor(P["card"])

bars = ax.barh(
    imp_df["Feature"],
    imp_df["Importance"],
    color=P["accent"],
    edgecolor=P["border"],
    height=0.65
)

for bar, val in zip(bars, imp_df["Importance"]):
    ax.text(
        bar.get_width() + 0.001,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.3f}",
        va="center",
        fontsize=8,
        color=P["muted"]
    )

ax.set_xlabel("Importance")
ax.set_title(f"Feature Importance — {best_model_name} ({importance_source})_abc", color=P["text"], fontweight="bold")
ax.grid(axis="x")
plt.tight_layout()

fname = os.path.join(out_dir, f"feature_importance_{best_model_name.lower().replace(' ', '_')}_abc.png")
fig.savefig(fname, dpi=150, bbox_inches="tight", facecolor=P["bg"])
plt.close(fig)
print(f"Saved: {fname}")

# =============================================================================
# FINAL SUMMARY
# =============================================================================
print("\n" + "="*55)
print("IMPROVEMENTS APPLIED")
print("  1. KNN Imputer (k=5, distance-weighted) instead of median/mode")
print("  2. TARGET_N raised 200 → 300 (more augmented samples)")
print("  3. abc fitness: Accuracy → AUC  (more discriminative)")
print("  4. abc wasps/iters: 10/12 → 15/20 (Stage1), 8/10 → 12/15 (Stage2)")
print("  5. BorderlineSMOTE instead of standard SMOTE")
print("  6. RobustScaler on continuous features (critical for KNN)")
print("  7. Threshold: F1-only → Youden's J (sensitivity + specificity - 1)")
print("  8. RandomizedSearchCV n_iter: 60 → 100 + wider hyperparameter grids")
print("  9. RepeatedStratifiedKFold (5×3) for threshold optimisation")
print("="*55)
print("DONE")
print(f"  Real train / test split : {len(X_train_r)} / {len(X_test_r)}")
print(f"  Augmented training size : {len(X_aug)}")
print(f"  Final features          : {len(final_features)}")
print(f"\n  -- Test-set results (real held-out patients, Youden threshold) --")
for name, r in results.items():
    mark = "  [BEST AUC]" if name == best_name else ""
    print(f"  {name:<18}  Acc={r['accuracy']:.3f}  AUC={r['auc']:.3f}  "
          f"Rec={r['recall']:.3f}  F1={r['f1']:.3f}  thresh={r['threshold']:.2f}{mark}")
print()